# Business Entity Resolution — full self-contained pipeline (Kaggle, Run-All)

**What this does.** From the RAW competition data (attached under `/kaggle/input`) this single notebook runs the whole matching pipeline end-to-end and writes the two deliverable TSVs to `/kaggle/working`: `matching_results.tsv` and `candidate_pairs.tsv`.

Stages: clean/normalize -> per-country blocking index -> 20k-India labeled dev set -> 37 pairwise features -> train LightGBM+XGBoost blend -> score every test S1 (India/US/France) at the frozen operating threshold **t\*=0.98** -> validate format.

**How to run.** (1) Add the competition data as an input (the six `*_source*.tsv` files + `train_ground_truth.tsv`). (2) Settings -> Accelerator: **None (CPU)**; **Internet: On** (only to `pip install` the similarity libraries — no external entity data is ever fetched). (3) **Run All**, or *Save Version* to run headless (CPU kernels allow up to ~12 h).

**This is NOT a competition submission.** It only writes local TSVs to the kernel output; `kaggle competitions submit` is never called. No external data lookup resolves entities (challenge fair-play); romanization uses a permissively-licensed fallback (anyascii / stdlib), not GPL unidecode.

**Honest accuracy note.** The measured leakage-safe CV of this exact blend is macro-F0.5 ~ **0.85** on the 20k-India dev set (trained on India, applied to US/France out-of-distribution). This notebook reproduces that config; it does **not** claim 0.956/0.98, which sit above the blocking recall ceiling.


In [ ]:
# --- deps: LIBRARIES only (never external entity data) ----------------------
# If Internet is OFF and any import below is missing, turn Internet ON (Settings)
# and re-run this cell. Installing PyPI wheels is not an "external data lookup".
import sys, subprocess
for _pkg in ("rapidfuzz", "jellyfish", "anyascii", "unidecode"):
    try:
        __import__(_pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)
import numpy, pandas, sklearn, lightgbm, xgboost   # already in the Kaggle image
print("deps ok:", numpy.__version__, pandas.__version__,
      lightgbm.__version__, xgboost.__version__)


In [ ]:
# --- lay out /tmp/ber (ROOT), discover raw inputs, write the pipeline modules
import os, sys, glob, json, base64, shutil
ROOT = "/tmp/ber"                       # ROOT -> src/../../.. ; big intermediates stay out of the 20GB output
SRC_DIR = ROOT + "/code/business_entity_resolution/src"
for d in (ROOT + "/dataset/train", ROOT + "/dataset/test", SRC_DIR):
    os.makedirs(d, exist_ok=True)

def find_raw(*pats):
    for p in pats:
        hits = [h for h in sorted(glob.glob("/kaggle/input/**/" + p, recursive=True))
                if os.path.isfile(h)]
        if hits:
            return hits[0]
    return None

RAW = {
    "train/train_source1.tsv": find_raw("train_source1.tsv", "*train*source*1*.tsv"),
    "train/train_source2.tsv": find_raw("train_source2.tsv", "*train*source*2*.tsv"),
    "train/train_source3.tsv": find_raw("train_source3.tsv", "*train*source*3*.tsv"),
    "test/test_source1.tsv":   find_raw("test_source1.tsv", "*test*source*1*.tsv"),
    "test/test_source2.tsv":   find_raw("test_source2.tsv", "*test*source*2*.tsv"),
    "test/test_source3.tsv":   find_raw("test_source3.tsv", "*test*source*3*.tsv"),
    "train/train_ground_truth.tsv": find_raw("train_ground_truth.tsv", "*ground_truth*.tsv"),
}
missing = [k for k, v in RAW.items() if not v]
assert not missing, ("attach the competition data; not found: " + str(missing) +
                     "  |  /kaggle/input/* = " + str(glob.glob("/kaggle/input/*")))
for rel, srcpath in RAW.items():
    dst = ROOT + "/dataset/" + rel
    if os.path.lexists(dst):
        os.remove(dst)
    try:
        os.symlink(srcpath, dst)
    except Exception:
        shutil.copy(srcpath, dst)
    print(rel, "<-", srcpath, flush=True)

FILES = json.loads(r"""{"common.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTaGFyZWQgdXRpbGl0aWVzIGZvciB0aGUgQnVzaW5lc3MgRW50aXR5IFJlc29sdXRpb24gbWF0Y2hpbmcgcGlwZWxpbmUuCgpQYXRocywgc2NoZW1hIGNvbnN0YW50cywgY2h1bmtlZCBUU1YgSU8sIHRoZSBtYWNyby1GMC41IHNjb3JlciwgbGVha2FnZS1zYWZlCkdyb3VwS0ZvbGQgYXNzaWdubWVudCAoZ3JvdXBlZCBieSBTb3VyY2UtMSBlbnRpdHkpLCBjYW5kaWRhdGUtcGFpciBJTywgYW5kCmxpZ2h0d2VpZ2h0IHRva2VuaXphdGlvbiBoZWxwZXJzLiBPZmZsaW5lIG9ubHkuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKaW1wb3J0IHN5cwpmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgSXRlcmFibGUsIExpc3QsIFNldCwgVHVwbGUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgp0cnk6CiAgICBzeXMuc3Rkb3V0LnJlY29uZmlndXJlKGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpCiAgICBzeXMuc3RkZXJyLnJlY29uZmlndXJlKGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpCmV4Y2VwdCBFeGNlcHRpb246CiAgICBwYXNzCgpIRVJFID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpClJPT1QgPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKEhFUkUsICIuLiIsICIuLiIsICIuLiIpKQoKIyBFdmVyeSBwYXRoIGNhbiBiZSBvdmVycmlkZGVuIGJ5IGFuIGVudiB2YXIgc28gdGhlIFNBTUUgY29kZSBydW5zIHVuY2hhbmdlZCBib3RoCiMgbG9jYWxseSAocmVwbyBsYXlvdXQpIGFuZCBvbiBLYWdnbGUgKHJlYWQtb25seSAva2FnZ2xlL2lucHV0IGZvciBDTEVBTi9NT0RFTFMsCiMgd3JpdGFibGUgL2thZ2dsZS93b3JraW5nIGZvciBXT1JLL09VVFBVVCkuIERlZmF1bHRzIHJlcHJvZHVjZSB0aGUgcmVwbyBsYXlvdXQuCkNMRUFOID0gb3MuZW52aXJvbi5nZXQoIkJFUl9DTEVBTiIpIG9yIG9zLnBhdGguam9pbihST09ULCAiZGF0YXNldCIsICJjbGVhbiIpCldPUksgPSBvcy5lbnZpcm9uLmdldCgiQkVSX1dPUksiKSBvciBvcy5wYXRoLmpvaW4oUk9PVCwgImRhdGFzZXQiLCAid29yayIpICAjIGxhcmdlIGludGVybWVkaWF0ZXMKQ0FORF9ESVIgPSBvcy5wYXRoLmpvaW4oV09SSywgImNhbmRpZGF0ZXMiKQpGRUFUX0RJUiA9IG9zLnBhdGguam9pbihXT1JLLCAiZmVhdHVyZXMiKQpPT0ZfRElSID0gb3MucGF0aC5qb2luKFdPUkssICJvb2YiKQpNT0RFTFMgPSBvcy5lbnZpcm9uLmdldCgiQkVSX01PREVMUyIpIG9yIG9zLnBhdGguam9pbihST09ULCAibW9kZWxzIikgICAgICAgIyBnaXRpZ25vcmVkCk9VVFBVVCA9IG9zLmVudmlyb24uZ2V0KCJCRVJfT1VUUFVUIikgb3Igb3MucGF0aC5qb2luKFJPT1QsICJvdXRwdXQiKQpSRVBPUlRfRElSID0gb3MucGF0aC5qb2luKE9VVFBVVCwgIm1hdGNoaW5nIikKCiMgT25seSB0aGUgd3JpdGFibGUgZGlycyBhcmUgYXV0by1jcmVhdGVkOyBDTEVBTi9NT0RFTFMgYXJlIHJlYWQtb25seSBpbnB1dHMgb24gS2FnZ2xlLgpmb3IgX2QgaW4gKFdPUkssIENBTkRfRElSLCBGRUFUX0RJUiwgT09GX0RJUiwgT1VUUFVULCBSRVBPUlRfRElSKToKICAgIHRyeToKICAgICAgICBvcy5tYWtlZGlycyhfZCwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgIHBhc3MKdHJ5OgogICAgb3MubWFrZWRpcnMoTU9ERUxTLCBleGlzdF9vaz1UcnVlKQpleGNlcHQgT1NFcnJvcjoKICAgIHBhc3MKCkNMRUFOX0NPTFMgPSBbImVudGl0eV9pZCIsICJjb3VudHJ5IiwgImJ1c2luZXNzX25hbWUiLCAiYnVzaW5lc3NfYWRkcmVzcyIsCiAgICAgICAgICAgICAgIm5hbWVfbm9ybSIsICJuYW1lX2NvcmUiLCAibmFtZV9zdWZmaXgiLCAiYWRkcl9ub3JtIl0KCiMgY2xlYW5lZCBzb3VyY2UgZmlsZSAtPiByb2xlClRSQUlOX0ZJTEVTID0geyJTMSI6IG9zLnBhdGguam9pbihDTEVBTiwgInRyYWluIiwgInRyYWluX3NvdXJjZTEudHN2IiksCiAgICAgICAgICAgICAgICJTMiI6IG9zLnBhdGguam9pbihDTEVBTiwgInRyYWluIiwgInRyYWluX3NvdXJjZTIudHN2IiksCiAgICAgICAgICAgICAgICJTMyI6IG9zLnBhdGguam9pbihDTEVBTiwgInRyYWluIiwgInRyYWluX3NvdXJjZTMudHN2Iil9ClRFU1RfRklMRVMgPSB7IlMxIjogb3MucGF0aC5qb2luKENMRUFOLCAidGVzdCIsICJ0ZXN0X3NvdXJjZTEudHN2IiksCiAgICAgICAgICAgICAgIlMyIjogb3MucGF0aC5qb2luKENMRUFOLCAidGVzdCIsICJ0ZXN0X3NvdXJjZTIudHN2IiksCiAgICAgICAgICAgICAgIlMzIjogb3MucGF0aC5qb2luKENMRUFOLCAidGVzdCIsICJ0ZXN0X3NvdXJjZTMudHN2Iil9CkdST1VORF9UUlVUSCA9IG9zLnBhdGguam9pbihST09ULCAiZGF0YXNldCIsICJ0cmFpbiIsICJ0cmFpbl9ncm91bmRfdHJ1dGgudHN2IikKCiMgUExBQ0VIT0xERVJfSU8KCgpkZWYgcmVhZF9jbGVhbihwYXRoOiBzdHIsIHVzZWNvbHM6IExpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICBucm93czogaW50IHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkxvYWQgYSBjbGVhbmVkIFRTViAoYWxsIHN0cmluZyBkdHlwZSwgTkEtc2FmZSkuIiIiCiAgICBkZiA9IHBkLnJlYWRfY3N2KHBhdGgsIHNlcD0iXHQiLCBkdHlwZT1zdHIsIGtlZXBfZGVmYXVsdF9uYT1GYWxzZSwgbmFfdmFsdWVzPVtdLAogICAgICAgICAgICAgICAgICAgICBxdW90aW5nPTMsIGVuY29kaW5nPSJ1dGYtOCIsIGVuY29kaW5nX2Vycm9ycz0icmVwbGFjZSIsCiAgICAgICAgICAgICAgICAgICAgIHVzZWNvbHM9dXNlY29scywgbnJvd3M9bnJvd3MpCiAgICByZXR1cm4gZGYuZmlsbG5hKCIiKQoKCmRlZiBpdGVyX2NsZWFuKHBhdGg6IHN0ciwgY2h1bmtzaXplOiBpbnQgPSA0MDBfMDAwLCB1c2Vjb2xzOiBMaXN0W3N0cl0gfCBOb25lID0gTm9uZSk6CiAgICAiIiJTdHJlYW0gYSBjbGVhbmVkIFRTViBpbiBjaHVua3MgKG1lbW9yeS1zYWZlKS4iIiIKICAgIHJlYWRlciA9IHBkLnJlYWRfY3N2KHBhdGgsIHNlcD0iXHQiLCBkdHlwZT1zdHIsIGtlZXBfZGVmYXVsdF9uYT1GYWxzZSwgbmFfdmFsdWVzPVtdLAogICAgICAgICAgICAgICAgICAgICAgICAgcXVvdGluZz0zLCBlbmNvZGluZz0idXRmLTgiLCBlbmNvZGluZ19lcnJvcnM9InJlcGxhY2UiLAogICAgICAgICAgICAgICAgICAgICAgICAgdXNlY29scz11c2Vjb2xzLCBjaHVua3NpemU9Y2h1bmtzaXplKQogICAgZm9yIGNodW5rIGluIHJlYWRlcjoKICAgICAgICB5aWVsZCBjaHVuay5maWxsbmEoIiIpCgoKZGVmIGxvYWRfZ3JvdW5kX3RydXRoKCkgLT4gRGljdFtzdHIsIFNldFtzdHJdXToKICAgICIiIlMxIGVudGl0eV9pZCAtPiBzZXQgb2YgbWF0Y2hlZCBTMi9TMyBpZHMgKHt9IG1lYW5zIHNpbmdsZXRvbikuIiIiCiAgICBndDogRGljdFtzdHIsIFNldFtzdHJdXSA9IHt9CiAgICByZWFkZXIgPSBwZC5yZWFkX2NzdihHUk9VTkRfVFJVVEgsIHNlcD0iXHQiLCBkdHlwZT1zdHIsIGtlZXBfZGVmYXVsdF9uYT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgIG5hX3ZhbHVlcz1bXSwgcXVvdGluZz0zLCBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICAgICAgICAgICAgICAgZW5jb2RpbmdfZXJyb3JzPSJyZXBsYWNlIiwgY2h1bmtzaXplPTQwMF8wMDApCiAgICBmb3IgY2h1bmsgaW4gcmVhZGVyOgogICAgICAgIGNodW5rID0gY2h1bmsuZmlsbG5hKCIiKQogICAgICAgIGZvciBzMSwgbWF0Y2hlZCBpbiB6aXAoY2h1bmtbInNvdXJjZTFfZW50aXR5X2lkIl0sIGNodW5rWyJtYXRjaGVkX2VudGl0eV9pZHMiXSk6CiAgICAgICAgICAgIGd0W3MxXSA9IHt4IGZvciB4IGluIHN0cihtYXRjaGVkKS5zcGxpdCgiLCIpIGlmIHh9CiAgICByZXR1cm4gZ3QKCgojIC0tLSBDYW5kaWRhdGUtcGFpciBJTzogb25lIHJvdyBwZXIgUzEgZW50aXR5LCBjb21tYS1qb2luZWQgY2FuZGlkYXRlIGlkcyAtLS0tLQpkZWYgd3JpdGVfY2FuZGlkYXRlX2ZpbGUocGF0aDogc3RyLCBjYW5kczogRGljdFtzdHIsIExpc3Rbc3RyXV0sCiAgICAgICAgICAgICAgICAgICAgICAgICBoZWFkZXJfY29sOiBzdHIgPSAiY2FuZGlkYXRlX2VudGl0eV9pZHMiKSAtPiBOb25lOgogICAgIiIiV3JpdGUge3MxX2lkOiBbaWRzXX0gdG8gYSBzdWJtaXNzaW9uLXN0eWxlIFRTViAoZW1wdHkgbGlzdCAtPiBlbXB0eSBjZWxsKS4iIiIKICAgIHJvd3MgPSBbKHMxLCAiLCIuam9pbihpZHMpKSBmb3IgczEsIGlkcyBpbiBjYW5kcy5pdGVtcygpXQogICAgZGYgPSBwZC5EYXRhRnJhbWUocm93cywgY29sdW1ucz1bInNvdXJjZTFfZW50aXR5X2lkIiwgaGVhZGVyX2NvbF0pCiAgICBkZi50b19jc3YocGF0aCwgc2VwPSJcdCIsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgiLCBsaW5ldGVybWluYXRvcj0iXG4iKQoKCmRlZiByZWFkX2NhbmRpZGF0ZV9maWxlKHBhdGg6IHN0cikgLT4gRGljdFtzdHIsIExpc3Rbc3RyXV06CiAgICAiIiJSZWFkIGEgc3VibWlzc2lvbi1zdHlsZSBUU1YgYmFjayBpbnRvIHtzMV9pZDogW2lkc119LiIiIgogICAgb3V0OiBEaWN0W3N0ciwgTGlzdFtzdHJdXSA9IHt9CiAgICBkZiA9IHBkLnJlYWRfY3N2KHBhdGgsIHNlcD0iXHQiLCBkdHlwZT1zdHIsIGtlZXBfZGVmYXVsdF9uYT1GYWxzZSwgbmFfdmFsdWVzPVtdLAogICAgICAgICAgICAgICAgICAgICBxdW90aW5nPTMsIGVuY29kaW5nPSJ1dGYtOCIsIGVuY29kaW5nX2Vycm9ycz0icmVwbGFjZSIpCiAgICB2YWxfY29sID0gZGYuY29sdW1uc1sxXQogICAgZm9yIHMxLCB2IGluIHppcChkZlsic291cmNlMV9lbnRpdHlfaWQiXSwgZGZbdmFsX2NvbF0pOgogICAgICAgIG91dFtzMV0gPSBbeCBmb3IgeCBpbiBzdHIodikuc3BsaXQoIiwiKSBpZiB4XQogICAgcmV0dXJuIG91dAoKIyBQTEFDRUhPTERFUl9TQ09SRQoKCmRlZiBmYmV0YV9lbnRpdHkocHJlZDogU2V0W3N0cl0sIHRydXRoOiBTZXRbc3RyXSwgYmV0YTogZmxvYXQgPSAwLjUpIC0+IGZsb2F0OgogICAgIiIiRl9iZXRhIGZvciBvbmUgUzEgZW50aXR5LiBFbXB0eS10cnV0aDogMS4wIGlmIHByZWQgZW1wdHkgZWxzZSAwLjAuIiIiCiAgICBpZiBub3QgdHJ1dGg6CiAgICAgICAgcmV0dXJuIDEuMCBpZiBub3QgcHJlZCBlbHNlIDAuMAogICAgaWYgbm90IHByZWQ6CiAgICAgICAgcmV0dXJuIDAuMAogICAgdHAgPSBsZW4ocHJlZCAmIHRydXRoKQogICAgaWYgdHAgPT0gMDoKICAgICAgICByZXR1cm4gMC4wCiAgICBwcmVjaXNpb24gPSB0cCAvIGxlbihwcmVkKQogICAgcmVjYWxsID0gdHAgLyBsZW4odHJ1dGgpCiAgICBiMiA9IGJldGEgKiBiZXRhCiAgICBkZW5vbSA9IGIyICogcHJlY2lzaW9uICsgcmVjYWxsCiAgICByZXR1cm4gKDEgKyBiMikgKiBwcmVjaXNpb24gKiByZWNhbGwgLyBkZW5vbSBpZiBkZW5vbSBlbHNlIDAuMAoKCmRlZiBtYWNyb19mYmV0YShwcmVkczogRGljdFtzdHIsIFNldFtzdHJdXSwgdHJ1dGhzOiBEaWN0W3N0ciwgU2V0W3N0cl1dLAogICAgICAgICAgICAgICAgYmV0YTogZmxvYXQgPSAwLjUpIC0+IFR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdOgogICAgIiIiTWFjcm8gRl9iZXRhIG92ZXIgYWxsIFMxIGVudGl0aWVzIGluIGB0cnV0aHNgLiBSZXR1cm5zIChGLCBtZWFuUCwgbWVhblIpLgoKICAgIEFueSBTMSBpbiBgdHJ1dGhzYCBtaXNzaW5nIGZyb20gYHByZWRzYCBpcyB0cmVhdGVkIGFzIHByZWRpY3RpbmcgZW1wdHkuCiAgICAiIiIKICAgIGZzLCBwcywgcnMgPSBbXSwgW10sIFtdCiAgICBmb3IgczEsIHRydXRoIGluIHRydXRocy5pdGVtcygpOgogICAgICAgIHByZWQgPSBwcmVkcy5nZXQoczEsIHNldCgpKQogICAgICAgIGZzLmFwcGVuZChmYmV0YV9lbnRpdHkocHJlZCwgdHJ1dGgsIGJldGEpKQogICAgICAgIGlmIHRydXRoOgogICAgICAgICAgICB0cCA9IGxlbihwcmVkICYgdHJ1dGgpCiAgICAgICAgICAgIHBzLmFwcGVuZCh0cCAvIGxlbihwcmVkKSBpZiBwcmVkIGVsc2UgMC4wKQogICAgICAgICAgICBycy5hcHBlbmQodHAgLyBsZW4odHJ1dGgpKQogICAgcmV0dXJuIChmbG9hdChucC5tZWFuKGZzKSkgaWYgZnMgZWxzZSAwLjAsCiAgICAgICAgICAgIGZsb2F0KG5wLm1lYW4ocHMpKSBpZiBwcyBlbHNlIDAuMCwKICAgICAgICAgICAgZmxvYXQobnAubWVhbihycykpIGlmIHJzIGVsc2UgMC4wKQoKCmRlZiBhc3NpZ25fZm9sZHMoczFfaWRzOiBJdGVyYWJsZVtzdHJdLCBrOiBpbnQgPSA1LCBzZWVkOiBpbnQgPSA0MikgLT4gRGljdFtzdHIsIGludF06CiAgICAiIiJEZXRlcm1pbmlzdGljIGZvbGQgaWQgcGVyIFMxIGVudGl0eSAobGVha2FnZS1zYWZlIGdyb3VwaW5nIHVuaXQpLgoKICAgIFVzZXMgYSBTVEFCTEUgaGFzaCAobWQ1KSBzbyB0aGUgU0FNRSBzMV9pZCBhbHdheXMgbGFuZHMgaW4gdGhlIHNhbWUgZm9sZAogICAgYWNyb3NzIHNlcGFyYXRlIHByb2Nlc3MgcnVucyDigJQgUHl0aG9uJ3MgYnVpbHRpbiBoYXNoKCkgaXMgc2FsdGVkIHBlciBwcm9jZXNzCiAgICAoUFlUSE9OSEFTSFNFRUQpIGFuZCB3b3VsZCBnaXZlIGRpZmZlcmVudCBmb2xkcyBlYWNoIHJ1biwgc2lsZW50bHkgYnJlYWtpbmcKICAgIHRoZSB0d28tbGV2ZWwgT09GIHN0YWNraW5nIChiYXNlLW1vZGVsIE9PRiBhbmQgdGhlIGZ1c2lvbiBtZXRhLWxlYXJuZXIgTVVTVAogICAgc2hhcmUgb25lIGlkZW50aWNhbCBmb2xkIGFzc2lnbm1lbnQpLgogICAgIiIiCiAgICBpbXBvcnQgaGFzaGxpYgogICAgZm9sZHM6IERpY3Rbc3RyLCBpbnRdID0ge30KICAgIGZvciBzMSBpbiBzMV9pZHM6CiAgICAgICAgaCA9IGhhc2hsaWIubWQ1KGYie3NlZWR9OntzMX0iLmVuY29kZSgidXRmLTgiKSkuaGV4ZGlnZXN0KCkKICAgICAgICBmb2xkc1tzMV0gPSBpbnQoaCwgMTYpICUgawogICAgcmV0dXJuIGZvbGRzCgoKIyAtLS0gVG9rZW5pemF0aW9uIGhlbHBlcnMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIHRva2VucyhzOiBzdHIpIC0+IExpc3Rbc3RyXToKICAgIHJldHVybiBzLnNwbGl0KCkgaWYgcyBlbHNlIFtdCgoKZGVmIG51bWVyaWNfdG9rZW5zKHM6IHN0cikgLT4gU2V0W3N0cl06CiAgICAiIiJEaWdpdC1iZWFyaW5nIHRva2VucyAoc3RyZWV0IG51bWJlcnMsIFBJTiBjb2Rlcykg4oCUIHN0cm9uZyBhZGRyZXNzIHNpZ25hbC4iIiIKICAgIHJldHVybiB7dCBmb3IgdCBpbiBzLnNwbGl0KCkgaWYgYW55KGMuaXNkaWdpdCgpIGZvciBjIGluIHQpfQoKCmRlZiBjaGFyX25ncmFtcyhzOiBzdHIsIG46IGludCA9IDMpIC0+IFNldFtzdHJdOgogICAgcyA9IHMucmVwbGFjZSgiICIsICIiKQogICAgcmV0dXJuIHtzW2k6aSArIG5dIGZvciBpIGluIHJhbmdlKGxlbihzKSAtIG4gKyAxKX0gaWYgbGVuKHMpID49IG4gZWxzZSAoe3N9IGlmIHMgZWxzZSBzZXQoKSkKCgo=", "blocking.py": "...", "features.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTdGFnZSAyIOKAlCBQYWlyd2lzZSBmZWF0dXJlcyBmb3IgKFMxLCBjYW5kaWRhdGUpIHBhaXJzLgoKQ29uc3VtZXMgYSBkZXZzZXQvY2FuZGlkYXRlIHBhcnF1ZXQgd2l0aCB0aGUgZm91ciB0ZXh0IGZpZWxkcyBpbmxpbmUKKHMxX2NvcmUsIHMxX2FkZHIsIGNfY29yZSwgY19hZGRyKSBwbHVzIHMxX2lkL2NhbmRfaWQvbGFiZWwvZm9sZCwgYW5kIGVtaXRzIGEKZmVhdHVyZSBtYXRyaXggcGFycXVldC4gQWxsIGZlYXR1cmVzIGFyZSBwYWlyd2lzZSBURVhUIHNpbWlsYXJpdGllcyArIGEKbGFiZWwtZnJlZSBnZW5lcmljLW5hbWUgSURGIHNpZ25hbCDigJQgbm8gZmVhdHVyZSB1c2VzIG1hdGNoIHN0cnVjdHVyZSBvciByZWNvcmQKaWRlbnRpdHksIHNvIGZpdHRpbmcgdGhlIElERiBvbiB0aGUgZnVsbCBjb3JwdXMgaXMgbGVha2FnZS1zYWZlICh0cmFuc2R1Y3RpdmUpLgoKRmVhdHVyZSBmYW1pbGllczoKICBuYW1lICA6IHJhcGlkZnV6eiByYXRpb3MsIHRva2VuL2NoYXItbmdyYW0gSmFjY2FyZCwgamFyby13aW5rbGVyLCBleGFjdCwKICAgICAgICAgIGNvbmNhdGVuYXRpb24gKHNwYWNlLXN0cmlwcGVkIGNvbnRhaW5tZW50KSwgbGVuZ3RoL3Rva2VuLWNvdW50IHJhdGlvcwogIGlkZW50IDogc2hhcmVkLXRva2VuIGNvdW50ICsgcmFyZXN0LXNoYXJlZC10b2tlbiBJREYsIHMxIG5hbWUgZ2VuZXJpYy1uZXNzCiAgYWRkciAgOiByYXBpZGZ1enogcmF0aW9zLCBudW1lcmljLXRva2VuIChzdHJlZXQjL1BJTikgb3ZlcmxhcCwgbG9jYWxpdHkKICAgICAgICAgIEphY2NhcmQsIHByZXNlbmNlIGZsYWdzLCBjaGFyLW5ncmFtIEphY2NhcmQKICBjcm9zcyA6IG5hbWUgeCBhZGRyZXNzIGludGVyYWN0aW9uIChnZW5lcmljIG5hbWVzIG5lZWQgYWRkcmVzcyBhZ3JlZW1lbnQpCgpSdW46ICBweXRob24gZmVhdHVyZXMucHkgPGRldnNldC5wYXJxdWV0PiBbLS1vdXQgcGF0aF0gWy0tY2h1bmsgNTAwMDAwXQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBvcwppbXBvcnQgcGlja2xlCmltcG9ydCB0aW1lCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHJhcGlkZnV6eiBpbXBvcnQgZnV6egppbXBvcnQgamVsbHlmaXNoCgppbXBvcnQgY29tbW9uIGFzIGMKCklERl9DQUNIRSA9IG9zLnBhdGguam9pbihjLldPUkssICJ0b2tkZl9pbmRpYS5wa2wiKQpOVU1JREZfQ0FDSEUgPSBvcy5wYXRoLmpvaW4oYy5XT1JLLCAibnVtaWRmX2luZGlhLnBrbCIpCkZFQVRVUkVTID0gWwogICAgIm5fcmF0aW8iLCAibl90c29ydCIsICJuX3RzZXQiLCAibl9wYXJ0aWFsIiwgIm5fd3JhdGlvIiwgIm5fanciLAogICAgIm5fdG9rX2phYyIsICJuX2czX2phYyIsICJuX2c0X2phYyIsICJuX2V4YWN0IiwgIm5fY29uY2F0IiwKICAgICJuX2xlbl9yYXRpbyIsICJuX250b2tfcmF0aW8iLAogICAgImlkX3NoYXJlZF90b2siLCAiaWRfc2hhcmVfbWF4aWRmIiwgImlkX3N1bV9pZGYiLCAiczFfbWluX2lkZiIsICJzMV9tYXhfaWRmIiwKICAgICJhX3RzZXQiLCAiYV90c29ydCIsICJhX2c0X2phYyIsICJhX251bV9vdiIsICJhX251bV9qYWMiLCAiYV9sb2NfamFjIiwKICAgICJhX2xvbmdudW1fb3YiLCAiYV9udW1fbWF4aWRmIiwgImFfbG9jX292IiwKICAgICJhX2JvdGgiLCAiYV9zMV9lbXB0eSIsICJhX2NfZW1wdHkiLAogICAgImFfbnVtX2NvbmZsaWN0IiwgImFfbG9uZ251bV9jb25mbGljdCIsICJuX3VuaXFfbWF4aWRmIiwgIm5fdW5pcV9zdW1pZGYiLAogICAgInhfbmFtZV9hZGRyIiwgInhfbmFtZV9udW0iLCAiaXNfczMiLApdCgoKZGVmIF9pZGYoKToKICAgIHdpdGggb3BlbihJREZfQ0FDSEUsICJyYiIpIGFzIGY6CiAgICAgICAgcmV0dXJuIHBpY2tsZS5sb2FkKGYpCgoKZGVmIF9udW1pZGYoKToKICAgICIiIk51bWVyaWMtYWRkcmVzcy10b2tlbiBJREYgKHJhcmUgYnVpbGRpbmcvUElOIG1hdGNoKS4ge30gaWYgbm90IGJ1aWx0IHlldC4iIiIKICAgIGlmIG9zLnBhdGguZXhpc3RzKE5VTUlERl9DQUNIRSk6CiAgICAgICAgd2l0aCBvcGVuKE5VTUlERl9DQUNIRSwgInJiIikgYXMgZjoKICAgICAgICAgICAgcmV0dXJuIHBpY2tsZS5sb2FkKGYpCiAgICByZXR1cm4ge30KCgpkZWYgX2phYyhhOiBzZXQsIGI6IHNldCkgLT4gZmxvYXQ6CiAgICBpZiBub3QgYSBhbmQgbm90IGI6CiAgICAgICAgcmV0dXJuIDAuMAogICAgdSA9IGxlbihhIHwgYikKICAgIHJldHVybiBsZW4oYSAmIGIpIC8gdSBpZiB1IGVsc2UgMC4wCgoKZGVmIF9yZWdpb24oYWRkcjogc3RyKSAtPiBzZXQ6CiAgICByZXR1cm4ge3QgZm9yIHQgaW4gYWRkci5zcGxpdCgpIGlmIGxlbih0KSA+PSA0IGFuZCBub3QgYW55KGNoLmlzZGlnaXQoKSBmb3IgY2ggaW4gdCl9CgoKZGVmIGNvbXB1dGVfY2h1bmsoZGY6IHBkLkRhdGFGcmFtZSwgaWRmOiBkaWN0LCBudW1pZGY6IGRpY3QgfCBOb25lID0gTm9uZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQ29tcHV0ZSB0aGUgRkVBVFVSRVMgbWF0cml4IGZvciBvbmUgY2h1bms7IHJldHVybnMgYSBmbG9hdDMyIERhdGFGcmFtZS4iIiIKICAgIG51bWlkZiA9IG51bWlkZiBvciB7fQogICAgbiA9IGxlbihkZikKICAgIG91dCA9IHtmOiBucC56ZXJvcyhuLCBkdHlwZT1ucC5mbG9hdDMyKSBmb3IgZiBpbiBGRUFUVVJFU30KICAgIHMxYyA9IGRmWyJzMV9jb3JlIl0udG9fbnVtcHkoKTsgczFhID0gZGZbInMxX2FkZHIiXS50b19udW1weSgpCiAgICBjYyA9IGRmWyJjX2NvcmUiXS50b19udW1weSgpOyAgY2EgPSBkZlsiY19hZGRyIl0udG9fbnVtcHkoKQogICAgY2lkID0gKGRmWyJjYW5kX2lkIl0udG9fbnVtcHkoKSBpZiAiY2FuZF9pZCIgaW4gZGYuY29sdW1ucwogICAgICAgICAgIGVsc2UgbnAuYXJyYXkoWyIiXSAqIG4sIGR0eXBlPW9iamVjdCkpCiAgICBtZWRfaWRmID0gbnAubG9nKDRfMTMzXzM0NiAvIDUwLjApICAgICAgICAgICMgZmFsbGJhY2sgaWRmIGZvciB1bnNlZW4gbmFtZSB0b2tlbnMKICAgIG1lZF9udW1pZGYgPSBucC5sb2coNF8xMzNfMzQ2IC8gMjAuMCkgICAgICAgIyBmYWxsYmFjayBpZGYgZm9yIHVuc2VlbiBudW1lcmljIHRva2VucwogICAgZm9yIGkgaW4gcmFuZ2Uobik6CiAgICAgICAgYSwgYiA9IHMxY1tpXSwgY2NbaV0KICAgICAgICAjIC0tLSBuYW1lIC0tLQogICAgICAgIG91dFsibl9yYXRpbyJdW2ldID0gZnV6ei5yYXRpbyhhLCBiKQogICAgICAgIG91dFsibl90c29ydCJdW2ldID0gZnV6ei50b2tlbl9zb3J0X3JhdGlvKGEsIGIpCiAgICAgICAgb3V0WyJuX3RzZXQiXVtpXSA9IGZ1enoudG9rZW5fc2V0X3JhdGlvKGEsIGIpCiAgICAgICAgb3V0WyJuX3BhcnRpYWwiXVtpXSA9IGZ1enoucGFydGlhbF9yYXRpbyhhLCBiKQogICAgICAgIG91dFsibl93cmF0aW8iXVtpXSA9IGZ1enouV1JhdGlvKGEsIGIpCiAgICAgICAgb3V0WyJuX2p3Il1baV0gPSBqZWxseWZpc2guamFyb193aW5rbGVyX3NpbWlsYXJpdHkoYSwgYikgKiAxMDAuMAogICAgICAgIHRhLCB0YiA9IHNldChhLnNwbGl0KCkpLCBzZXQoYi5zcGxpdCgpKQogICAgICAgIG91dFsibl90b2tfamFjIl1baV0gPSBfamFjKHRhLCB0YikKICAgICAgICBvdXRbIm5fZzNfamFjIl1baV0gPSBfamFjKGMuY2hhcl9uZ3JhbXMoYSwgMyksIGMuY2hhcl9uZ3JhbXMoYiwgMykpCiAgICAgICAgb3V0WyJuX2c0X2phYyJdW2ldID0gX2phYyhjLmNoYXJfbmdyYW1zKGEsIDQpLCBjLmNoYXJfbmdyYW1zKGIsIDQpKQogICAgICAgIG91dFsibl9leGFjdCJdW2ldID0gMS4wIGlmIGEgPT0gYiBhbmQgYSBlbHNlIDAuMAogICAgICAgIHNhLCBzYiA9IGEucmVwbGFjZSgiICIsICIiKSwgYi5yZXBsYWNlKCIgIiwgIiIpCiAgICAgICAgb3V0WyJuX2NvbmNhdCJdW2ldID0gMS4wIGlmIHNhIGFuZCBzYiBhbmQgKHNhIGluIHNiIG9yIHNiIGluIHNhKSBlbHNlIDAuMAogICAgICAgIGxhLCBsYiA9IGxlbihhKSwgbGVuKGIpCiAgICAgICAgb3V0WyJuX2xlbl9yYXRpbyJdW2ldID0gbWluKGxhLCBsYikgLyBtYXgobGEsIGxiKSBpZiBtYXgobGEsIGxiKSBlbHNlIDAuMAogICAgICAgIG5hLCBuYiA9IGxlbih0YSksIGxlbih0YikKICAgICAgICBvdXRbIm5fbnRva19yYXRpbyJdW2ldID0gbWluKG5hLCBuYikgLyBtYXgobmEsIG5iKSBpZiBtYXgobmEsIG5iKSBlbHNlIDAuMAogICAgICAgICMgLS0tIGlkZW50aXR5IC8gZ2VuZXJpY25lc3MgLS0tCiAgICAgICAgc2hhcmVkID0gdGEgJiB0YgogICAgICAgIG91dFsiaWRfc2hhcmVkX3RvayJdW2ldID0gbGVuKHNoYXJlZCkKICAgICAgICBzaGFyZWRfaWRmcyA9IFtpZGYuZ2V0KHQsIG1lZF9pZGYpIGZvciB0IGluIHNoYXJlZF0KICAgICAgICBvdXRbImlkX3NoYXJlX21heGlkZiJdW2ldID0gbWF4KHNoYXJlZF9pZGZzLCBkZWZhdWx0PTAuMCkKICAgICAgICBvdXRbImlkX3N1bV9pZGYiXVtpXSA9IGZsb2F0KHN1bShzaGFyZWRfaWRmcykpICAgICAjIHRvdGFsIGRpc3RpbmN0aXZlIGV2aWRlbmNlCiAgICAgICAgczFpZGZzID0gW2lkZi5nZXQodCwgbWVkX2lkZikgZm9yIHQgaW4gdGFdCiAgICAgICAgb3V0WyJzMV9taW5faWRmIl1baV0gPSBtaW4oczFpZGZzKSBpZiBzMWlkZnMgZWxzZSAwLjAKICAgICAgICBvdXRbInMxX21heF9pZGYiXVtpXSA9IG1heChzMWlkZnMpIGlmIHMxaWRmcyBlbHNlIDAuMAogICAgICAgICMgLS0tIGFkZHJlc3MgLS0tCiAgICAgICAgYWEsIGFiID0gczFhW2ldLCBjYVtpXQogICAgICAgIG91dFsiYV90c2V0Il1baV0gPSBmdXp6LnRva2VuX3NldF9yYXRpbyhhYSwgYWIpCiAgICAgICAgb3V0WyJhX3Rzb3J0Il1baV0gPSBmdXp6LnRva2VuX3NvcnRfcmF0aW8oYWEsIGFiKQogICAgICAgIG91dFsiYV9nNF9qYWMiXVtpXSA9IF9qYWMoYy5jaGFyX25ncmFtcyhhYSwgNCksIGMuY2hhcl9uZ3JhbXMoYWIsIDQpKQogICAgICAgIG51bWEsIG51bWIgPSBjLm51bWVyaWNfdG9rZW5zKGFhKSwgYy5udW1lcmljX3Rva2VucyhhYikKICAgICAgICBzaGFyZWRfbnVtID0gbnVtYSAmIG51bWIKICAgICAgICBvdXRbImFfbnVtX292Il1baV0gPSBsZW4oc2hhcmVkX251bSkKICAgICAgICBvdXRbImFfbnVtX2phYyJdW2ldID0gX2phYyhudW1hLCBudW1iKQogICAgICAgICMgcmFyZS1udW1lcmljIG1hdGNoOiBhIHNoYXJlZCBidWlsZGluZy9mbGF0L1BJTiBudW1iZXIgaXMgaGlnaGx5IGRpYWdub3N0aWMKICAgICAgICBvdXRbImFfbG9uZ251bV9vdiJdW2ldID0gc3VtKDEgZm9yIHQgaW4gc2hhcmVkX251bSBpZiBsZW4odCkgPj0gNCkKICAgICAgICBvdXRbImFfbnVtX21heGlkZiJdW2ldID0gbWF4KChudW1pZGYuZ2V0KHQsIG1lZF9udW1pZGYpIGZvciB0IGluIHNoYXJlZF9udW0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGVmYXVsdD0wLjApCiAgICAgICAgcmVnYSwgcmVnYiA9IF9yZWdpb24oYWEpLCBfcmVnaW9uKGFiKQogICAgICAgIG91dFsiYV9sb2NfamFjIl1baV0gPSBfamFjKHJlZ2EsIHJlZ2IpCiAgICAgICAgb3V0WyJhX2xvY19vdiJdW2ldID0gbGVuKHJlZ2EgJiByZWdiKQogICAgICAgIG91dFsiYV9ib3RoIl1baV0gPSAxLjAgaWYgYWEgYW5kIGFiIGVsc2UgMC4wCiAgICAgICAgb3V0WyJhX3MxX2VtcHR5Il1baV0gPSAxLjAgaWYgbm90IGFhIGVsc2UgMC4wCiAgICAgICAgb3V0WyJhX2NfZW1wdHkiXVtpXSA9IDEuMCBpZiBub3QgYWIgZWxzZSAwLjAKICAgICAgICAjIC0tLSBjb25mbGljdCAvIHZldG8gc2lnbmFscyAoYXR0YWNrIHNhdHVyYXRlZC1hZ3JlZW1lbnQgZmFsc2UgbWVyZ2VzKSAtLS0KICAgICAgICAjIFRoZXNlIGFyZSBESVNBR1JFRU1FTlQgZmVhdHVyZXM6IHRoZSAzMyBiYXNlbGluZSBmZWF0cyBhcmUgYWxtb3N0IGFsbAogICAgICAgICMgYWdyZWVtZW50IHNpZ25hbHMsIGJ1dCB0aGUgcmVzaWR1YWwgT09GIGZhbHNlLXBvc2l0aXZlcyBhcmUgaGFyZCBuZWFyLQogICAgICAgICMgZHVwbGljYXRlcyBhbHJlYWR5IHNhdHVyYXRlZCBvbiBhZ3JlZW1lbnQuIEEgY29uZmxpY3Rpbmcgc3RyZWV0L1BJTiBudW1iZXIKICAgICAgICAjIG9yIGEgZGlzdGluY3RpdmUgbmFtZSB0b2tlbiBwcmVzZW50IG9uIG9ubHkgb25lIHNpZGUgaXMgc3Ryb25nIGV2aWRlbmNlIG9mCiAgICAgICAgIyB0d28gRElGRkVSRU5UIGVudGl0aWVzIC0+IGEgdmV0byBheGlzIHRoZSBHQkRUIGxhY2tlZC4KICAgICAgICBvdXRbImFfbnVtX2NvbmZsaWN0Il1baV0gPSAxLjAgaWYgKG51bWEgYW5kIG51bWIgYW5kIG5vdCBzaGFyZWRfbnVtKSBlbHNlIDAuMAogICAgICAgIGxvbmdhID0ge3QgZm9yIHQgaW4gbnVtYSBpZiBsZW4odCkgPj0gNH0KICAgICAgICBsb25nYiA9IHt0IGZvciB0IGluIG51bWIgaWYgbGVuKHQpID49IDR9CiAgICAgICAgb3V0WyJhX2xvbmdudW1fY29uZmxpY3QiXVtpXSA9IDEuMCBpZiAobG9uZ2EgYW5kIGxvbmdiIGFuZCBub3QgKGxvbmdhICYgbG9uZ2IpKSBlbHNlIDAuMAogICAgICAgIHVuaXEgPSB0YSBeIHRiICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgdG9rZW5zIG9uIGV4YWN0bHkgb25lIHNpZGUKICAgICAgICB1bmlxX2lkZnMgPSBbaWRmLmdldCh0LCBtZWRfaWRmKSBmb3IgdCBpbiB1bmlxXQogICAgICAgIG91dFsibl91bmlxX21heGlkZiJdW2ldID0gbWF4KHVuaXFfaWRmcywgZGVmYXVsdD0wLjApCiAgICAgICAgb3V0WyJuX3VuaXFfc3VtaWRmIl1baV0gPSBmbG9hdChzdW0odW5pcV9pZGZzKSkKICAgICAgICAjIC0tLSBjcm9zcyAvIHNvdXJjZSAtLS0KICAgICAgICBvdXRbInhfbmFtZV9hZGRyIl1baV0gPSAob3V0WyJuX3RzZXQiXVtpXSAvIDEwMC4wKSAqIChvdXRbImFfdHNldCJdW2ldIC8gMTAwLjApCiAgICAgICAgb3V0WyJ4X25hbWVfbnVtIl1baV0gPSAob3V0WyJuX3RzZXQiXVtpXSAvIDEwMC4wKSAqIChtaW4ob3V0WyJhX251bV9vdiJdW2ldLCAyKSAvIDIuMCkKICAgICAgICBvdXRbImlzX3MzIl1baV0gPSAxLjAgaWYgc3RyKGNpZFtpXSkuc3RhcnRzd2l0aCgiUzMiKSBlbHNlIDAuMAogICAgcmVzID0gcGQuRGF0YUZyYW1lKG91dCkKICAgIGZvciBrZWVwIGluICgiczFfaWQiLCAiY2FuZF9pZCIsICJsYWJlbCIsICJmb2xkIik6CiAgICAgICAgaWYga2VlcCBpbiBkZi5jb2x1bW5zOgogICAgICAgICAgICByZXNba2VlcF0gPSBkZltrZWVwXS50b19udW1weSgpCiAgICByZXR1cm4gcmVzCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCJkZXZzZXQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jaHVuayIsIHR5cGU9aW50LCBkZWZhdWx0PTUwMF8wMDApCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgaWRmID0gX2lkZigpCiAgICBudW1pZGYgPSBfbnVtaWRmKCkKICAgIGlmIG5vdCBudW1pZGY6CiAgICAgICAgcHJpbnQoIiAgW3dhcm5dIG51bWlkZl9pbmRpYS5wa2wgbWlzc2luZyDigJQgYV9udW1fbWF4aWRmIHVzZXMgZmFsbGJhY2sgaWRmOyAiCiAgICAgICAgICAgICAgInJ1biBidWlsZF9udW1pZGYucHkgZm9yIHRoZSByYXJlLW51bWVyaWMgc2lnbmFsIiwgZmx1c2g9VHJ1ZSkKICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KGFyZ3MuZGV2c2V0KQogICAgb3V0ID0gYXJncy5vdXQgb3IgYXJncy5kZXZzZXQucmVwbGFjZSgiLnBhcnF1ZXQiLCAiX2ZlYXRzLnBhcnF1ZXQiKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgcGFydHMgPSBbXQogICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihkZiksIGFyZ3MuY2h1bmspOgogICAgICAgIHN1YiA9IGRmLmlsb2Nbc3RhcnQ6c3RhcnQgKyBhcmdzLmNodW5rXQogICAgICAgIHBhcnRzLmFwcGVuZChjb21wdXRlX2NodW5rKHN1YiwgaWRmLCBudW1pZGYpKQogICAgICAgIGRvbmUgPSBtaW4oc3RhcnQgKyBhcmdzLmNodW5rLCBsZW4oZGYpKQogICAgICAgIGVsID0gdGltZS50aW1lKCkgLSB0MAogICAgICAgIHByaW50KGYiICB7ZG9uZTosfS97bGVuKGRmKTosfSBwYWlycyAgKHtkb25lL2VsOiwuMGZ9L3MpIiwgZmx1c2g9VHJ1ZSkKICAgIGZlYXRzID0gcGQuY29uY2F0KHBhcnRzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgIGZlYXRzLnRvX3BhcnF1ZXQob3V0LCBpbmRleD1GYWxzZSkKICAgIHByaW50KGYiZmVhdHVyZXM6IHtsZW4oZmVhdHMpOix9IHJvd3MgeCB7bGVuKEZFQVRVUkVTKX0gZmVhdHMgaW4ge3RpbWUudGltZSgpLXQwOi4wZn1zICIKICAgICAgICAgIGYiKHtsZW4oZmVhdHMpLyh0aW1lLnRpbWUoKS10MCk6LC4wZn0gcGFpcnMvcykgLT4ge291dH0iLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "make_devset.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWlsZCBhIHNlbGYtY29udGFpbmVkIGxhYmVsZWQgZGV2IHNldCBmb3IgdGhlIG1hdGNoaW5nIG1vZGVsIChJbmRpYSBzaGFyZCkuCgpSZXVzZXMgdGhlIGNhY2hlZCBJbmRpYSBibG9ja2luZyBpbmRleCAoaWR4X2luZGlhX2NhcDMwMDAucGtsKSBzbyB3ZSBET04nVApyZS1zdHJlYW0gdGhlIDI0TSBjbGVhbmVkIHJvd3MuIEZvciB0aGUgZmlyc3QgTiBJbmRpYSBTMSBlbnRpdGllczoKICAtIGdlbmVyYXRlIGNhbmRpZGF0ZXMgKGNoZWFwLW9ubHkgcmFua2luZywgbG9ja2VkIEZBTV9XLCB0b3BrKSwKICAtIHB1bGwgZWFjaCBjYW5kaWRhdGUncyBuYW1lX2NvcmUvYWRkcl9ub3JtIHN0cmFpZ2h0IGZyb20gdGhlIGluZGV4IGFycmF5cywKICAtIGxhYmVsIDEgaWYgdGhlIGNhbmRpZGF0ZSBpZCBpcyBpbiBncm91bmQgdHJ1dGggZm9yIHRoYXQgUzEgZWxzZSAwLAogIC0gYXNzaWduIGEgbGVha2FnZS1zYWZlIEdyb3VwS0ZvbGQgaWQgKGdyb3VwZWQgYnkgUzEgZW50aXR5KSwKYW5kIHdyaXRlIG9uZSBwYXJxdWV0IHdpdGggZXZlcnl0aGluZyBpbmxpbmUuIEFsc28gZW1pdHMgYSBnbG9iYWwgbmFtZS10b2tlbgpJREYgKGRvY3VtZW50IGZyZXF1ZW5jeSBvdmVyIEFMTCBTMitTMyBJbmRpYSBuYW1lX2NvcmUsIGluY2wuIGdlbmVyaWMgdG9rZW5zCnRoZSBibG9ja2luZyBpbmRleCBkcm9wcykgZm9yIHRoZSBnZW5lcmljLW5hbWUgZmVhdHVyZS4gU2luZ2xldG9uIFMxIChlbXB0eQp0cnV0aCkgQVJFIGluY2x1ZGVkIOKAlCB0aGUgbW9kZWwgbXVzdCBsZWFybiB0byBlbWl0IGVtcHR5IGZvciB0aGVtLgoKUnVuOiAgcHl0aG9uIG1ha2VfZGV2c2V0LnB5IFtOXSAgIChkZWZhdWx0IE49MjAwMDApCiIiIgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgcGlja2xlCmltcG9ydCB0aW1lCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgppbXBvcnQgY29tbW9uIGFzIGMKaW1wb3J0IGJsb2NraW5nIGFzIGJsCgpCVUlMRF9DQVAgPSAzMDAwCkNBQ0hFID0gb3MucGF0aC5qb2luKGMuV09SSywgZiJpZHhfaW5kaWFfY2Fwe0JVSUxEX0NBUH0ucGtsIikKSURGX0NBQ0hFID0gb3MucGF0aC5qb2luKGMuV09SSywgInRva2RmX2luZGlhLnBrbCIpClRPUEsgPSAyMDAgICAgICAgICAgICAgICAgICAgICAgIyBsZW5ub3JtIGNlaWxpbmcgfjAuOTQzIEAgMjBrICh3YXMgMTUwIHJhdy11bmlvbiB+MC45MzQpCkNPVU5UUlkgPSAiSW5kaWEiCgoKZGVmIGxvYWRfaW5kZXgoKToKICAgIHQwID0gdGltZS50aW1lKCkKICAgIHdpdGggb3BlbihDQUNIRSwgInJiIikgYXMgZjoKICAgICAgICBjaSA9IHBpY2tsZS5sb2FkKGYpCiAgICBwcmludChmImxvYWRlZCBjYWNoZWQge0NPVU5UUll9IGluZGV4ICh7Y2kubl9yZWNvcmRzKCk6LH0gcmVjcykgaW4ge3RpbWUudGltZSgpLXQwOi4wZn1zIiwKICAgICAgICAgIGZsdXNoPVRydWUpCiAgICByZXR1cm4gY2kKCgpkZWYgdG9rZW5faWRmKGNpKSAtPiBkaWN0OgogICAgIiIiR2xvYmFsIG5hbWUtdG9rZW4gSURGIG92ZXIgdGhlIGZ1bGwgaW5kZXggY29yZXMgKGluY2wuIGdlbmVyaWMgdG9rZW5zKS4iIiIKICAgIGlmIG9zLnBhdGguZXhpc3RzKElERl9DQUNIRSk6CiAgICAgICAgd2l0aCBvcGVuKElERl9DQUNIRSwgInJiIikgYXMgZjoKICAgICAgICAgICAgcmV0dXJuIHBpY2tsZS5sb2FkKGYpCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBkZiA9IENvdW50ZXIoKQogICAgZm9yIGNvcmUgaW4gY2kuY29yZXM6CiAgICAgICAgZm9yIHQgaW4gc2V0KGNvcmUuc3BsaXQoKSk6CiAgICAgICAgICAgIGRmW3RdICs9IDEKICAgIG4gPSBjaS5uX3JlY29yZHMoKQogICAgaWRmID0ge3Q6IGZsb2F0KG5wLmxvZyhuIC8gZCkpIGZvciB0LCBkIGluIGRmLml0ZW1zKCl9CiAgICB3aXRoIG9wZW4oSURGX0NBQ0hFLCAid2IiKSBhcyBmOgogICAgICAgIHBpY2tsZS5kdW1wKGlkZiwgZiwgcHJvdG9jb2w9NCkKICAgIHByaW50KGYiYnVpbHQgdG9rZW4gaWRmICh7bGVuKGlkZik6LH0gdG9rZW5zKSBpbiB7dGltZS50aW1lKCktdDA6LjBmfXMiLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGlkZgoKCmRlZiBtYWluKCk6CiAgICBuX3RhcmdldCA9IGludChzeXMuYXJndlsxXSkgaWYgbGVuKHN5cy5hcmd2KSA+IDEgZWxzZSAyMDAwMAogICAgY2kgPSBsb2FkX2luZGV4KCkKICAgIGNpLmVuc3VyZV9ub3JtKCkgICAgICAgICAgICAgICAgICAgICAgICAjIGNvc2luZS1hcHByb3ggY2hlYXAgcmFua2VyIChyZWNhbGwgd2luKQogICAgXyA9IHRva2VuX2lkZihjaSkgICAgICAgICAgICAgICAgICAgICAgICMgc2lkZS1lZmZlY3Q6IGNhY2hlIElERiBmb3IgZmVhdHVyZXMucHkKCiAgICBndCA9IGMubG9hZF9ncm91bmRfdHJ1dGgoKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgcm93cyA9IFtdICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgKHMxX2lkLCBjYW5kX2lkLCBsYWJlbCwgczFfY29yZSwgczFfYWRkciwgY19jb3JlLCBjX2FkZHIpCiAgICBzMV9zZWVuID0gW10gICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBvcmRlcmVkIHVuaXF1ZSBzMSBpZHMgZm9yIGZvbGQgYXNzaWdubWVudAogICAgbiA9IDAKICAgIGZvciBjaHVuayBpbiBjLml0ZXJfY2xlYW4oYy5UUkFJTl9GSUxFU1siUzEiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdXNlY29scz1bImVudGl0eV9pZCIsICJjb3VudHJ5IiwgIm5hbWVfY29yZSIsICJhZGRyX25vcm0iXSk6CiAgICAgICAgc3ViID0gY2h1bmtbY2h1bmtbImNvdW50cnkiXS52YWx1ZXMgPT0gQ09VTlRSWV0KICAgICAgICBmb3IgcmlkLCBjb3JlLCBhZGRyIGluIHppcChzdWJbImVudGl0eV9pZCJdLCBzdWJbIm5hbWVfY29yZSJdLCBzdWJbImFkZHJfbm9ybSJdKToKICAgICAgICAgICAgcG9zID0gYmwuY2FuZGlkYXRlc19mb3IoY2ksIGNvcmUsIGFkZHIsIFRPUEssIGVmZl9jYXA9QlVJTERfQ0FQLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZXRhaWxfY2FwPTAsIHJldHVybl9wb3M9VHJ1ZSkKICAgICAgICAgICAgdHJ1dGggPSBndC5nZXQocmlkLCBzZXQoKSkKICAgICAgICAgICAgczFfc2Vlbi5hcHBlbmQocmlkKQogICAgICAgICAgICBmb3IgaiBpbiBwb3M6CiAgICAgICAgICAgICAgICBjaWQgPSBjaS5pZHNbal0KICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKChyaWQsIGNpZCwgMSBpZiBjaWQgaW4gdHJ1dGggZWxzZSAwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvcmUsIGFkZHIsIGNpLmNvcmVzW2pdLCBjaS5hZGRyc1tqXSkpCiAgICAgICAgICAgIG4gKz0gMQogICAgICAgICAgICBpZiBuID49IG5fdGFyZ2V0OgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBuID49IG5fdGFyZ2V0OgogICAgICAgICAgICBicmVhawoKICAgIGZvbGRzID0gYy5hc3NpZ25fZm9sZHMoczFfc2Vlbiwgaz01LCBzZWVkPTQyKQogICAgZGYgPSBwZC5EYXRhRnJhbWUocm93cywgY29sdW1ucz1bInMxX2lkIiwgImNhbmRfaWQiLCAibGFiZWwiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInMxX2NvcmUiLCAiczFfYWRkciIsICJjX2NvcmUiLCAiY19hZGRyIl0pCiAgICBkZlsiZm9sZCJdID0gZGZbInMxX2lkIl0ubWFwKGZvbGRzKS5hc3R5cGUoImludDgiKQogICAgZGZbImxhYmVsIl0gPSBkZlsibGFiZWwiXS5hc3R5cGUoImludDgiKQogICAgb3V0ID0gb3MucGF0aC5qb2luKGMuV09SSywgZiJkZXZzZXRfaW5kaWFfbntuX3RhcmdldH0ucGFycXVldCIpCiAgICBkZi50b19wYXJxdWV0KG91dCwgaW5kZXg9RmFsc2UpCgogICAgbnBvcyA9IGludChkZlsibGFiZWwiXS5zdW0oKSkKICAgIG5zaW5nID0gc3VtKDEgZm9yIHMgaW4gczFfc2VlbiBpZiBub3QgZ3QuZ2V0KHMpKQogICAgcHJpbnQoZiJkZXZzZXQ6IHtsZW4oczFfc2Vlbik6LH0gUzEgKHtuc2luZzosfSBzaW5nbGV0b25zKSwge2xlbihkZik6LH0gcGFpcnMsICIKICAgICAgICAgIGYie25wb3M6LH0gcG9zaXRpdmVzICh7bnBvcy9sZW4oZGYpOi4yJX0pLCAiCiAgICAgICAgICBmImF2ZyB7bGVuKGRmKS9sZW4oczFfc2Vlbik6LjFmfSBjYW5kL1MxLCBidWlsdCBpbiB7dGltZS50aW1lKCktdDA6LjBmfXMiLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoZiItPiB7b3V0fSIsIGZsdXNoPVRydWUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "build_numidf.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWlsZCBhIGdsb2JhbCBudW1lcmljLWFkZHJlc3MtdG9rZW4gSURGIChJbmRpYSBjb3JwdXMpIGZvciBmZWF0dXJlcyB2Mi4KClRoZSBzdHJvbmdlc3QgYWRkcmVzcyBzaWduYWwgaW4gdGhlIHBvc2l0aXZlIHBhaXJzIGlzIGEgc2hhcmVkICpyYXJlKiBudW1lcmljCnRva2VuIOKAlCBhIGJ1aWxkaW5nIC8gZmxhdCAvIFBJTiBudW1iZXIgbGlrZSAiMjUwNSIgdGhhdCBib3RoIHJlY29yZHMgY2FycnkuIFBsYWluCm92ZXJsYXAgY291bnQgKGFfbnVtX292KSB0cmVhdHMgYSBzaGFyZWQgIjEiIGxpa2UgYSBzaGFyZWQgIjcwMDA1MyI7IHdlaWdodGluZyBieQppbnZlcnNlIGRvY3VtZW50IGZyZXF1ZW5jeSBsZXRzIHRoZSBtb2RlbCByZXdhcmQgdGhlIHJhcmUsIGRpc2NyaW1pbmF0aXZlIG1hdGNoCmFuZCBpZ25vcmUgdGhlIHViaXF1aXRvdXMgb25lLgoKUmV1c2VzIHRoZSBjYWNoZWQgSW5kaWEgYmxvY2tpbmcgaW5kZXggKGl0cyAuYWRkcnMgYWxyZWFkeSBob2xkIGFkZHJfbm9ybSBmb3IgYWxsClMyK1MzIEluZGlhIHJlY29yZHMpIHNvIHdlIGRvbid0IHJlLXN0cmVhbSB0aGUgY29ycHVzLiBFbWl0cyBudW1pZGZfaW5kaWEucGtsOgp7IG51bWVyaWNfdG9rZW4gLT4gaWRmIH0gY29uc3VtZWQgYnkgZmVhdHVyZXMuY29tcHV0ZV9jaHVuay4KClJ1bjogIHB5dGhvbiBidWlsZF9udW1pZGYucHkKIiIiCmltcG9ydCBvcwppbXBvcnQgcGlja2xlCmltcG9ydCB0aW1lCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmltcG9ydCBudW1weSBhcyBucAoKaW1wb3J0IGNvbW1vbiBhcyBjCgpDQUNIRSA9IG9zLnBhdGguam9pbihjLldPUkssICJpZHhfaW5kaWFfY2FwMzAwMC5wa2wiKQpPVVQgPSBvcy5wYXRoLmpvaW4oYy5XT1JLLCAibnVtaWRmX2luZGlhLnBrbCIpCgoKZGVmIG1haW4oKToKICAgIHQwID0gdGltZS50aW1lKCkKICAgIHdpdGggb3BlbihDQUNIRSwgInJiIikgYXMgZjoKICAgICAgICBjaSA9IHBpY2tsZS5sb2FkKGYpCiAgICBuID0gbGVuKGNpLmFkZHJzKQogICAgZGYgPSBDb3VudGVyKCkKICAgIGZvciBhZGRyIGluIGNpLmFkZHJzOgogICAgICAgIGZvciB0IGluIGMubnVtZXJpY190b2tlbnMoc3RyKGFkZHIpKToKICAgICAgICAgICAgZGZbdF0gKz0gMQogICAgaWRmID0ge3Q6IGZsb2F0KG5wLmxvZyhuIC8gZCkpIGZvciB0LCBkIGluIGRmLml0ZW1zKCl9CiAgICB3aXRoIG9wZW4oT1VULCAid2IiKSBhcyBmOgogICAgICAgIHBpY2tsZS5kdW1wKGlkZiwgZiwgcHJvdG9jb2w9NCkKICAgIHByaW50KGYibnVtZXJpYyBpZGY6IHtsZW4oaWRmKTosfSB0b2tlbnMgb3ZlciB7bjosfSBhZGRycyAiCiAgICAgICAgICBmImluIHt0aW1lLnRpbWUoKS10MDouMGZ9cyAtPiB7T1VUfSIsIGZsdXNoPVRydWUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "cv_eval.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTdGFnZSAzKzUg4oCUIEJhc2UgbW9kZWxzIChMaWdodEdCTSB8fCBYR0Jvb3N0KSB3aXRoIGxlYWthZ2Utc2FmZSBHcm91cEtGb2xkIENWLgoKVHJhaW5zIExpZ2h0R0JNIGFuZCBYR0Jvb3N0IElOIFBBUkFMTEVMIG9uIHRoZSBzYW1lIHBhaXJ3aXNlIGZlYXR1cmUgbWF0cml4LApncm91cGVkIGJ5IFMxIGVudGl0eSAodGhlIG1hY3JvLUYwLjUgdW5pdCkgc28gbm8gZW50aXR5J3MgcGFpcnMgc3RyYWRkbGUgdGhlCnRyYWluL3ZhbCBzcGxpdC4gUHJvZHVjZXMgb3V0LW9mLWZvbGQgKE9PRikgcHJvYmFiaWxpdGllcyBmb3IgZXZlcnkgY2FuZGlkYXRlCnBhaXIg4oCUIHRoZSBob25lc3QgaW5wdXRzIGJvdGggdG8gdGhlIG1ldHJpYyBhbmQgdG8gdGhlIFN0YWdlLTQgZnVzaW9uIG5ldC4KCkxlYWthZ2UgY29udHJvbHM6CiAgKiBmb2xkID0gcHJlY29tcHV0ZWQgcGVyLVMxIEdyb3VwS0ZvbGQgaWQgKGFsbCBwYWlycyBvZiBhbiBTMSBzaGFyZSBhIGZvbGQpLgogICogbmVnYXRpdmVzIGFyZSBkb3duc2FtcGxlZCBvbiB0aGUgVFJBSU4gc2lkZSBvbmx5OyB0aGUgdmFsIGZvbGQgaXMgc2NvcmVkIG9uCiAgICBpdHMgRlVMTCBjYW5kaWRhdGUgc2V0LgogICogdGhlIGRlY2lzaW9uIHRocmVzaG9sZCBmb3IgYSBmb2xkIGlzIHR1bmVkIG9uIHRoZSBPVEhFUiBmb2xkcycgT09GLCBuZXZlciBvbgogICAgdGhlIGZvbGQgYmVpbmcgc2NvcmVkLgogICogbWFjcm8tRjAuNSBpcyBjb21wdXRlZCBhZ2FpbnN0IHRoZSBGVUxMIGdyb3VuZCB0cnV0aCAoaW5jbC4gbWF0Y2hlcyBibG9ja2luZwogICAgbmV2ZXIgc3VyZmFjZWQpLCBzbyB0aGUgbnVtYmVyIHJlZmxlY3RzIHRoZSByZWFsIHJlY2FsbCBjZWlsaW5nIOKAlCBub3QgYW4KICAgIGluZmxhdGVkIGluLWNhbmRpZGF0ZSByZWNhbGwuCgpFbWl0cyBvb2YgcGFycXVldCAoczFfaWQsIGNhbmRfaWQsIGxhYmVsLCBmb2xkLCBwX2xnYiwgcF94Z2IpIGZvciBmdXNpb24ucHkgYW5kCnByaW50cyBhIHBlci1mb2xkICsgbWVhbisvLXN0ZCBtYWNyby1GMC41L1AvUiB0YWJsZSBmb3IgTEdCIC8gWEdCIC8gYmxlbmQuCgpSdW46ICBweXRob24gY3ZfZXZhbC5weSA8ZmVhdHMucGFycXVldD4gWy0tbmVnIDNdIFstLW91dCBvb2YucGFycXVldF0KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgb3MKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgppbXBvcnQgY29tbW9uIGFzIGMKZnJvbSBmZWF0dXJlcyBpbXBvcnQgRkVBVFVSRVMKClRIUkVTSF9HUklEID0gbnAuY29uY2F0ZW5hdGUoW25wLmxpbnNwYWNlKDAuMDIsIDAuOTgsIDQ5KSwgWzAuOTg1LCAwLjk5LCAwLjk5NV1dKQoKCmRlZiBsZ2JfcGFyYW1zKCk6CiAgICByZXR1cm4gZGljdChuX2VzdGltYXRvcnM9NDAwLCBsZWFybmluZ19yYXRlPTAuMDUsIG51bV9sZWF2ZXM9NjMsCiAgICAgICAgICAgICAgICBtaW5fY2hpbGRfc2FtcGxlcz01MCwgc3Vic2FtcGxlPTAuOCwgc3Vic2FtcGxlX2ZyZXE9MSwKICAgICAgICAgICAgICAgIGNvbHNhbXBsZV9ieXRyZWU9MC44LCByZWdfbGFtYmRhPTEuMCwgbl9qb2JzPS0xLCB2ZXJib3NpdHk9LTEpCgoKZGVmIHhnYl9wYXJhbXMoKToKICAgIHJldHVybiBkaWN0KG5fZXN0aW1hdG9ycz00MDAsIGxlYXJuaW5nX3JhdGU9MC4wNSwgbWF4X2RlcHRoPTcsCiAgICAgICAgICAgICAgICBtaW5fY2hpbGRfd2VpZ2h0PTUsIHN1YnNhbXBsZT0wLjgsIGNvbHNhbXBsZV9ieXRyZWU9MC44LAogICAgICAgICAgICAgICAgcmVnX2xhbWJkYT0xLjAsIHRyZWVfbWV0aG9kPSJoaXN0Iiwgbl9qb2JzPS0xLCB2ZXJib3NpdHk9MCwKICAgICAgICAgICAgICAgIGV2YWxfbWV0cmljPSJsb2dsb3NzIikKCgpkZWYgX2Rvd25zYW1wbGUoaWR4OiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBuZWdfcmF0aW86IGZsb2F0LAogICAgICAgICAgICAgICAgc2VlZDogaW50KSAtPiBucC5uZGFycmF5OgogICAgIiIiS2VlcCBhbGwgcG9zaXRpdmVzICsgbmVnX3JhdGlvIHggdGhhdCBtYW55IG5lZ2F0aXZlcyAodHJhaW4gc2lkZSBvbmx5KS4iIiIKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgcG9zID0gaWR4W3lbaWR4XSA9PSAxXQogICAgbmVnID0gaWR4W3lbaWR4XSA9PSAwXQogICAgayA9IG1pbihsZW4obmVnKSwgaW50KGxlbihwb3MpICogbmVnX3JhdGlvKSkKICAgIG5lZyA9IHJuZy5jaG9pY2UobmVnLCBzaXplPWssIHJlcGxhY2U9RmFsc2UpCiAgICBvdXQgPSBucC5jb25jYXRlbmF0ZShbcG9zLCBuZWddKQogICAgcm5nLnNodWZmbGUob3V0KQogICAgcmV0dXJuIG91dAoKCmRlZiBmaXRfcHJlZGljdChYdHIsIHl0ciwgWHZhbCk6CiAgICAiIiJUcmFpbiBMR0IgYW5kIFhHQiBvbiB0aGUgc2FtZSByb3dzOyByZXR1cm4gKHBfbGdiX3ZhbCwgcF94Z2JfdmFsKS4iIiIKICAgIGltcG9ydCBsaWdodGdibSBhcyBsZ2IKICAgIGltcG9ydCB4Z2Jvb3N0IGFzIHhnYgogICAgbG0gPSBsZ2IuTEdCTUNsYXNzaWZpZXIoKipsZ2JfcGFyYW1zKCkpCiAgICBsbS5maXQoWHRyLCB5dHIpCiAgICBwX2xnYiA9IGxtLnByZWRpY3RfcHJvYmEoWHZhbClbOiwgMV0KICAgIHhtID0geGdiLlhHQkNsYXNzaWZpZXIoKip4Z2JfcGFyYW1zKCkpCiAgICB4bS5maXQoWHRyLCB5dHIpCiAgICBwX3hnYiA9IHhtLnByZWRpY3RfcHJvYmEoWHZhbClbOiwgMV0KICAgIHJldHVybiBwX2xnYiwgcF94Z2IKCgpkZWYgYnVpbGRfZ3JvdXBzKGRmOiBwZC5EYXRhRnJhbWUsIGd0OiBkaWN0KToKICAgICIiIlBlci1TMTogY2FuZGlkYXRlIGlkcywgdGhlaXIgcm93IGluZGljZXMsIGZ1bGwgdHJ1dGggc2V0LCBmb2xkLiIiIgogICAgZ3JwID0ge30KICAgIGZvciBzMSwgc3ViIGluIGRmLmdyb3VwYnkoInMxX2lkIiwgc29ydD1GYWxzZSk6CiAgICAgICAgZ3JwW3MxXSA9IGRpY3Qocm93cz1zdWIuaW5kZXgudG9fbnVtcHkoKSwKICAgICAgICAgICAgICAgICAgICAgICBjYW5kcz1zdWJbImNhbmRfaWQiXS50b19udW1weSgpLAogICAgICAgICAgICAgICAgICAgICAgIHRydXRoPWd0LmdldChzMSwgc2V0KCkpLAogICAgICAgICAgICAgICAgICAgICAgIGZvbGQ9aW50KHN1YlsiZm9sZCJdLmlsb2NbMF0pKQogICAgcmV0dXJuIGdycAoKCmRlZiBtYWNyb19mMDVfYXQoZ3JwLCBzMV9saXN0LCBwcm9iLCB0KToKICAgICIiIk1lYW4gRjAuNSBvdmVyIHMxX2xpc3QgYXQgdGhyZXNob2xkIHQgKGZ1bGwtdHJ1dGg7IGVtcHR5IHByZWQgYWxsb3dlZCkuIiIiCiAgICBmcyA9IFtdCiAgICBmb3IgczEgaW4gczFfbGlzdDoKICAgICAgICBnID0gZ3JwW3MxXQogICAgICAgIHByZWQgPSBzZXQoZ1siY2FuZHMiXVtwcm9iW2dbInJvd3MiXV0gPj0gdF0pCiAgICAgICAgZnMuYXBwZW5kKGMuZmJldGFfZW50aXR5KHByZWQsIGdbInRydXRoIl0sIDAuNSkpCiAgICByZXR1cm4gZmxvYXQobnAubWVhbihmcykpIGlmIGZzIGVsc2UgMC4wCgoKZGVmIGV2YWxfbW9kZWwoZ3JwLCBmb2xkc19vZiwgcHJvYiwgbmFtZSk6CiAgICAiIiJQZXItZm9sZCBtYWNyby1GMC41IHdpdGggdGhyZXNob2xkIHR1bmVkIG9uIHRoZSBPVEhFUiBmb2xkcycgT09GLiIiIgogICAgcGVyX2ZvbGQgPSBbXQogICAgZm9yIGYgaW4gcmFuZ2UoNSk6CiAgICAgICAgdmFsX3MxID0gZm9sZHNfb2ZbZl0KICAgICAgICB0dW5lX3MxID0gW3MgZm9yIGZmIGluIHJhbmdlKDUpIGlmIGZmICE9IGYgZm9yIHMgaW4gZm9sZHNfb2ZbZmZdXQogICAgICAgIGJlc3RfdCwgYmVzdCA9IDAuNSwgLTEuMAogICAgICAgIGZvciB0IGluIFRIUkVTSF9HUklEOgogICAgICAgICAgICB2ID0gbWFjcm9fZjA1X2F0KGdycCwgdHVuZV9zMSwgcHJvYiwgdCkKICAgICAgICAgICAgaWYgdiA+IGJlc3Q6CiAgICAgICAgICAgICAgICBiZXN0LCBiZXN0X3QgPSB2LCB0CiAgICAgICAgcGVyX2ZvbGQuYXBwZW5kKChtYWNyb19mMDVfYXQoZ3JwLCB2YWxfczEsIHByb2IsIGJlc3RfdCksIGJlc3RfdCkpCiAgICBmcyA9IG5wLmFycmF5KFtwWzBdIGZvciBwIGluIHBlcl9mb2xkXSkKICAgIHByaW50KGYiICB7bmFtZTo+N306IHBlci1mb2xkIEYwLjUgPSBbeycsICcuam9pbihmJ3t4Oi40Zn0nIGZvciB4IGluIGZzKX1dICAiCiAgICAgICAgICBmIm1lYW49e2ZzLm1lYW4oKTouNGZ9ICsvLSB7ZnMuc3RkKCk6LjRmfSAgKHR+e25wLm1lYW4oW3BbMV0gZm9yIHAgaW4gcGVyX2ZvbGRdKTouMmZ9KSIsCiAgICAgICAgICBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGZzCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCJmZWF0cyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbmVnIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zLjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD1Ob25lKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KGFyZ3MuZmVhdHMpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIGd0ID0gYy5sb2FkX2dyb3VuZF90cnV0aCgpCiAgICBYID0gZGZbRkVBVFVSRVNdLnRvX251bXB5KGR0eXBlPW5wLmZsb2F0MzIpCiAgICB5ID0gZGZbImxhYmVsIl0udG9fbnVtcHkoZHR5cGU9bnAuaW50OCkKICAgIGZvbGQgPSBkZlsiZm9sZCJdLnRvX251bXB5KGR0eXBlPW5wLmludDgpCgogICAgcF9sZ2IgPSBucC56ZXJvcyhsZW4oZGYpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgcF94Z2IgPSBucC56ZXJvcyhsZW4oZGYpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgZm9yIGYgaW4gcmFuZ2UoNSk6CiAgICAgICAgdHJfbWFzayA9IGZvbGQgIT0gZgogICAgICAgIHZhX2lkeCA9IG5wLndoZXJlKH50cl9tYXNrKVswXQogICAgICAgIHRyX2lkeCA9IF9kb3duc2FtcGxlKG5wLndoZXJlKHRyX21hc2spWzBdLCB5LCBhcmdzLm5lZywgc2VlZD0xMDAgKyBmKQogICAgICAgIHBsLCBweCA9IGZpdF9wcmVkaWN0KFhbdHJfaWR4XSwgeVt0cl9pZHhdLCBYW3ZhX2lkeF0pCiAgICAgICAgcF9sZ2JbdmFfaWR4XSA9IHBsCiAgICAgICAgcF94Z2JbdmFfaWR4XSA9IHB4CiAgICAgICAgcHJpbnQoZiIgIGZvbGQge2Z9OiB0cmFpbj17bGVuKHRyX2lkeCk6LH0gdmFsPXtsZW4odmFfaWR4KTosfSAiCiAgICAgICAgICAgICAgZiIoe3RpbWUudGltZSgpLXQwOi4wZn1zKSIsIGZsdXNoPVRydWUpCgogICAgZGZbInBfbGdiIl0gPSBwX2xnYgogICAgZGZbInBfeGdiIl0gPSBwX3hnYgogICAgcF9ibGVuZCA9IDAuNSAqIChwX2xnYiArIHBfeGdiKQogICAgb3V0ID0gYXJncy5vdXQgb3IgYXJncy5mZWF0cy5yZXBsYWNlKCJfZmVhdHMucGFycXVldCIsICJfb29mLnBhcnF1ZXQiKQogICAgZGZbWyJzMV9pZCIsICJjYW5kX2lkIiwgImxhYmVsIiwgImZvbGQiLCAicF9sZ2IiLCAicF94Z2IiXV0udG9fcGFycXVldChvdXQsIGluZGV4PUZhbHNlKQoKICAgIGdycCA9IGJ1aWxkX2dyb3VwcyhkZiwgZ3QpCiAgICBmb2xkc19vZiA9IHtmOiBbczEgZm9yIHMxLCBnIGluIGdycC5pdGVtcygpIGlmIGdbImZvbGQiXSA9PSBmXSBmb3IgZiBpbiByYW5nZSg1KX0KICAgIHByaW50KGYiXG5bY3ZdIHtsZW4oZ3JwKTosfSBTMSBlbnRpdGllcywgbWFjcm8tRjAuNSAoZnVsbC10cnV0aCwgbGVha2FnZS1zYWZlIHRocmVzaG9sZCk6IiwKICAgICAgICAgIGZsdXNoPVRydWUpCiAgICBldmFsX21vZGVsKGdycCwgZm9sZHNfb2YsIHBfbGdiLCAiTEdCIikKICAgIGV2YWxfbW9kZWwoZ3JwLCBmb2xkc19vZiwgcF94Z2IsICJYR0IiKQogICAgZXZhbF9tb2RlbChncnAsIGZvbGRzX29mLCBwX2JsZW5kLCAiYmxlbmQiKQogICAgcHJpbnQoZiJbY3ZdIG9vZiAtPiB7b3V0fSIsIGZsdXNoPVRydWUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "train.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJUcmFpbiB0aGUgRklOQUwgZGVwbG95IG1vZGVscyAoTEdCICsgWEdCKSBvbiB0aGUgZnVsbCBkZXYgZmVhdHVyZSBtYXRyaXguCgpjdl9ldmFsLnB5IHRyYWlucyBwZXItZm9sZCBmb3IgbGVha2FnZS1zYWZlIE9PRjsgdGhpcyBmaXRzIHRoZSBzYW1lIExHQi9YR0Igb24gQUxMCnJvd3MgKGV2ZXJ5IGZvbGQpIHNvIGluZmVyZW5jZSBnZXRzIG1heGltdW0gdHJhaW5pbmcgc2lnbmFsLCBhbmQgZnJlZXplcyBhIHNpbmdsZQpnbG9iYWwgb3BlcmF0aW5nIHRocmVzaG9sZCB0KiByZWFkIGZyb20gdGhlIGxlYWthZ2Utc2FmZSBPT0YuIFNhdmVzIGV2ZXJ5dGhpbmcgdGhlCmluZmVyZW5jZSBwYXNzIG5lZWRzIGludG8gbW9kZWxzLyA6CiAgZGVwbG95X2xnYi5wa2wsIGRlcGxveV94Z2IucGtsLCBkZXBsb3lfbWV0YS5qc29uIHtmZWF0dXJlcywgdGhyZXNob2xkLCBuZWcsIHByb2J9CgpSdW46ICBweXRob24gdHJhaW4ucHkgPGZlYXRzLnBhcnF1ZXQ+IC0tb29mIDxvb2YucGFycXVldD4gWy0tbmVnIDNdIFstLXByb2IgYmxlbmRdCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwaWNrbGUKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgppbXBvcnQgY29tbW9uIGFzIGMKZnJvbSBmZWF0dXJlcyBpbXBvcnQgRkVBVFVSRVMKZnJvbSBjdl9ldmFsIGltcG9ydCBsZ2JfcGFyYW1zLCB4Z2JfcGFyYW1zLCBfZG93bnNhbXBsZSwgYnVpbGRfZ3JvdXBzLCBtYWNyb19mMDVfYXQsIFRIUkVTSF9HUklECgoKZGVmIGRlcGxveV90aHJlc2hvbGQob29mX3BhdGg6IHN0ciwgcHJvYl9uYW1lOiBzdHIpIC0+IGZsb2F0OgogICAgIiIiT3BlcmF0aW5nIHQqOiBtYXhpbWl6ZSBtYWNyby1GMC41IG9uIHRoZSAobGVha2FnZS1zYWZlKSBPT0Ygb3ZlciBBTEwgZGV2IFMxLiIiIgogICAgb29mID0gcGQucmVhZF9wYXJxdWV0KG9vZl9wYXRoKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBndCA9IGMubG9hZF9ncm91bmRfdHJ1dGgoKQogICAgZ3JwID0gYnVpbGRfZ3JvdXBzKG9vZiwgZ3QpCiAgICBwcm9iID0geyJibGVuZCI6IDAuNSAqIChvb2ZbInBfbGdiIl0gKyBvb2ZbInBfeGdiIl0pLAogICAgICAgICAgICAibGdiIjogb29mWyJwX2xnYiJdLCAieGdiIjogb29mWyJwX3hnYiJdfVtwcm9iX25hbWVdLnRvX251bXB5KG5wLmZsb2F0MzIpCiAgICBhbGxfczEgPSBsaXN0KGdycCkKICAgIGJlc3RfdCwgYmVzdCA9IDAuNSwgLTEuMAogICAgZm9yIHQgaW4gVEhSRVNIX0dSSUQ6CiAgICAgICAgdiA9IG1hY3JvX2YwNV9hdChncnAsIGFsbF9zMSwgcHJvYiwgdCkKICAgICAgICBpZiB2ID4gYmVzdDoKICAgICAgICAgICAgYmVzdCwgYmVzdF90ID0gdiwgZmxvYXQodCkKICAgIHByaW50KGYiICBkZXBsb3kgdGhyZXNob2xkIHQqPXtiZXN0X3Q6LjNmfSAgKGRldiBtYWNyby1GMC41PXtiZXN0Oi40Zn0pIiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBiZXN0X3QKCgpkZWYgbWFpbigpOgogICAgaW1wb3J0IGxpZ2h0Z2JtIGFzIGxnYgogICAgaW1wb3J0IHhnYm9vc3QgYXMgeGdiCgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoImZlYXRzIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vb2YiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbmVnIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zLjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvYiIsIGRlZmF1bHQ9ImJsZW5kIiwgY2hvaWNlcz1bImJsZW5kIiwgImxnYiIsICJ4Z2IiXSkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBkZiA9IHBkLnJlYWRfcGFycXVldChhcmdzLmZlYXRzKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBYID0gZGZbRkVBVFVSRVNdLnRvX251bXB5KG5wLmZsb2F0MzIpCiAgICB5ID0gZGZbImxhYmVsIl0udG9fbnVtcHkobnAuaW50OCkKICAgIHQwID0gdGltZS50aW1lKCkKICAgIGlkeCA9IF9kb3duc2FtcGxlKG5wLmFyYW5nZShsZW4oZGYpKSwgeSwgYXJncy5uZWcsIHNlZWQ9NykKICAgIHByaW50KGYiW3RyYWluXSBmaXQgb24ge2xlbihpZHgpOix9IHJvd3MgIgogICAgICAgICAgZiIoe2ludCh5W2lkeF0uc3VtKCkpOix9IHBvcywgbmVnX3JhdGlvPXthcmdzLm5lZ30pIG9mIHtsZW4oZGYpOix9IiwgZmx1c2g9VHJ1ZSkKCiAgICBsbSA9IGxnYi5MR0JNQ2xhc3NpZmllcigqKmxnYl9wYXJhbXMoKSk7IGxtLmZpdChYW2lkeF0sIHlbaWR4XSkKICAgIHhtID0geGdiLlhHQkNsYXNzaWZpZXIoKip4Z2JfcGFyYW1zKCkpOyB4bS5maXQoWFtpZHhdLCB5W2lkeF0pCiAgICBwcmludChmIlt0cmFpbl0gbW9kZWxzIGZpdCBpbiB7dGltZS50aW1lKCktdDA6LjBmfXMiLCBmbHVzaD1UcnVlKQoKICAgIG9vZl9wYXRoID0gYXJncy5vb2Ygb3IgYXJncy5mZWF0cy5yZXBsYWNlKCJfZmVhdHMucGFycXVldCIsICJfb29mLnBhcnF1ZXQiKQogICAgdGhyID0gZGVwbG95X3RocmVzaG9sZChvb2ZfcGF0aCwgYXJncy5wcm9iKSBpZiBvcy5wYXRoLmV4aXN0cyhvb2ZfcGF0aCkgZWxzZSAwLjk4CgogICAgd2l0aCBvcGVuKG9zLnBhdGguam9pbihjLk1PREVMUywgImRlcGxveV9sZ2IucGtsIiksICJ3YiIpIGFzIGY6CiAgICAgICAgcGlja2xlLmR1bXAobG0sIGYpCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKGMuTU9ERUxTLCAiZGVwbG95X3hnYi5wa2wiKSwgIndiIikgYXMgZjoKICAgICAgICBwaWNrbGUuZHVtcCh4bSwgZikKICAgIG1ldGEgPSBkaWN0KGZlYXR1cmVzPUZFQVRVUkVTLCB0aHJlc2hvbGQ9dGhyLCBuZWc9YXJncy5uZWcsIHByb2I9YXJncy5wcm9iLAogICAgICAgICAgICAgICAgbl90cmFpbj1pbnQobGVuKGlkeCkpLCB0cmFpbmVkX29uPW9zLnBhdGguYmFzZW5hbWUoYXJncy5mZWF0cykpCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKGMuTU9ERUxTLCAiZGVwbG95X21ldGEuanNvbiIpLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG1ldGEsIGYsIGluZGVudD0yKQogICAgcHJpbnQoZiJbdHJhaW5dIHNhdmVkIGRlcGxveV9sZ2IveGdiICsgbWV0YSAodCo9e3RocjouM2Z9KSAtPiB7Yy5NT0RFTFN9IiwgZmx1c2g9VHJ1ZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "infer.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTdGFnZSA0IOKAlCBmdWxsLXRlc3QgaW5mZXJlbmNlIOKGkiB0aGUgdHdvIHN1Ym1pc3Npb24gVFNWcy4KClNjb3JlcyBFVkVSWSB0ZXN0IFMxIChhbGwgY291bnRyaWVzKSB3aXRoIHRoZSBmcm96ZW4gZGVwbG95IG1vZGVscyBhbmQgdGhlIHNpbmdsZQpnbG9iYWwgb3BlcmF0aW5nIHRocmVzaG9sZCB0KiBiYWtlZCBieSB0cmFpbi5weSwgdGhlbiB3cml0ZXM6CiAgb3V0cHV0L21hdGNoaW5nX3Jlc3VsdHMudHN2ICAgaGVhZGVyOiBzb3VyY2UxX2VudGl0eV9pZCBcdCBtYXRjaGVkX2VudGl0eV9pZHMKICBvdXRwdXQvY2FuZGlkYXRlX3BhaXJzLnRzdiAgICBoZWFkZXI6IHNvdXJjZTFfZW50aXR5X2lkIFx0IGNhbmRpZGF0ZV9lbnRpdHlfaWRzCkV2ZXJ5IHRlc3QgUzEgZ2V0cyBleGFjdGx5IG9uZSByb3c7IGFuIFMxIHdpdGggbm8gYWNjZXB0ZWQgbWF0Y2ggd3JpdGVzIGFuIGVtcHR5CmNlbGwgKGEgY29ycmVjdGx5LWVtcHR5IHNpbmdsZXRvbiBlYXJucyBmdWxsIEYwLjUgY3JlZGl0KS4KCk1lbW9yeSBwbGFuICgxNiBHQiBib3gsIH4zIEdCIGZyZWUpOgogIC0gT05FIGNvdW50cnkgYXQgYSB0aW1lOiBidWlsZCBpdHMgUzIrUzMgaW52ZXJ0ZWQgaW5kZXgsIGZyZWUgaXQgYmVmb3JlIHRoZSBuZXh0LgogIC0gc3RyZWFtIHRoYXQgY291bnRyeSdzIFMxIGluIGJhdGNoZXM7IHBlciBiYXRjaCwgZ2VuZXJhdGUgY2FuZGlkYXRlcyAoY2hlYXAKICAgIG51bXB5IHVuaW9uIHJhbmtpbmcpLCBjb21wdXRlIHRoZSAzMyBwYWlyd2lzZSBmZWF0dXJlcyBpbiBhIHByb2Nlc3MgcG9vbAogICAgKGlkZi9udW1pZGYgbG9hZGVkIG9uY2UgcGVyIHdvcmtlciksIGJsZW5kLXByZWRpY3QgTEdCK1hHQiwgdGhyZXNob2xkIGF0IHQqLAogICAgYW5kIHN0cmVhbS13cml0ZSBib3RoIFRTViByb3dzLiBOb3RoaW5nIGNvdW50cnktd2lkZSBpcyBoZWxkIGluIFJBTS4KCkJsb2NraW5nIE1BVENIRVMgbWFrZV9kZXZzZXQgKHRvcGs9MjAwLCBlZmZfY2FwPUJVSUxEX0NBUD0zMDAwLCBkZXRhaWxfY2FwPTApIHNvCmluZmVyZW5jZSBzZWVzIHRoZSBzYW1lIGNhbmRpZGF0ZSBkaXN0cmlidXRpb24gdGhlIG1vZGVscyB3ZXJlIHRyYWluZWQgb24uCgpOT1RFIChob25lc3QpOiB0aGUgZGVwbG95IG1vZGVscyArIGlkZi9udW1pZGYgd2VyZSBmaXQgb24gdGhlIEluZGlhIGRldiBzaGFyZC4KVVMgYW5kIEZyYW5jZSBhcmUgc2NvcmVkIG91dC1vZi1kaXN0cmlidXRpb24gd2l0aCB0aGUgc2FtZSBtb2RlbCBhbmQgSW5kaWEgaWRmCih1bnNlZW4gdG9rZW5zIGZhbGwgYmFjayB0byB0aGUgbWVkaWFuIGlkZikuIFRoaXMgaXMgdGhlICJiZXN0IGN1cnJlbnQgY29uZmlnIgphcHBsaWVkIHVuaWZvcm1seTsgcGVyLWNvdW50cnkgbW9kZWxzIGFyZSBhIGxhdGVyIHJvdW5kLgoKUnVuOiAgcHl0aG9uIGluZmVyLnB5IFstLXRvcGsgMjAwXSBbLS1iYXRjaC1zMSA0MDAwXSBbLS1wcm9jcyAxMF0gWy0tbGltaXQgTl0KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHBpY2tsZQppbXBvcnQgdGltZQpmcm9tIG11bHRpcHJvY2Vzc2luZyBpbXBvcnQgUG9vbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmltcG9ydCBjb21tb24gYXMgYwppbXBvcnQgYmxvY2tpbmcgYXMgYmwKaW1wb3J0IGZlYXR1cmVzIGFzIEYKCkJVSUxEX0NBUCA9IDMwMDAgICAgICAgICAgICAgICAgICAgICAgIyBkZl9jYXAgdXNlZCB0byBidWlsZCB0aGUgZGV2IGluZGV4IChtdXN0IG1hdGNoKQpJREZfUEFUSCA9IEYuSURGX0NBQ0hFICAgICAgICAgICAgICAgICAjIHRva2RmX2luZGlhLnBrbApOVU1JREZfUEFUSCA9IEYuTlVNSURGX0NBQ0hFICAgICAgICAgICAjIG51bWlkZl9pbmRpYS5wa2wKCiMgLS0tIHdvcmtlciAoZmVhdHVyZSBjb21wdXRhdGlvbiBvbmx5OyBubyBtb2RlbHMgbG9hZGVkIGhlcmUpIC0tLS0tLS0tLS0tLS0tLS0tCl9JREYgPSBOb25lCl9OVU1JREYgPSBOb25lCgoKZGVmIF93aW5pdChpZGZfcGF0aDogc3RyLCBudW1pZGZfcGF0aDogc3RyKSAtPiBOb25lOgogICAgIiIiUG9vbCBpbml0aWFsaXplcjogbG9hZCB0aGUgaWRmIGRpY3RzIG9uY2UgcGVyIHdvcmtlciBwcm9jZXNzLiIiIgogICAgZ2xvYmFsIF9JREYsIF9OVU1JREYKICAgIHdpdGggb3BlbihpZGZfcGF0aCwgInJiIikgYXMgZjoKICAgICAgICBfSURGID0gcGlja2xlLmxvYWQoZikKICAgIHRyeToKICAgICAgICB3aXRoIG9wZW4obnVtaWRmX3BhdGgsICJyYiIpIGFzIGY6CiAgICAgICAgICAgIF9OVU1JREYgPSBwaWNrbGUubG9hZChmKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBfTlVNSURGID0ge30KCgpkZWYgX3dmZWF0KGNodW5rOiBwZC5EYXRhRnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDb21wdXRlIHRoZSBGRUFUVVJFUyBtYXRyaXggZm9yIG9uZSBwYWlyIGNodW5rOyByZXR1cm4gZmxvYXQzMiAobiwgMzMpLiIiIgogICAgcmVzID0gRi5jb21wdXRlX2NodW5rKGNodW5rLCBfSURGLCBfTlVNSURGKQogICAgcmV0dXJuIHJlc1tGLkZFQVRVUkVTXS50b19udW1weShucC5mbG9hdDMyKQoKCiMgLS0tIGhlbHBlcnMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBfcHJlZGljdF9ibGVuZChsbSwgeG0sIFg6IG5wLm5kYXJyYXksIHByb2I6IHN0cikgLT4gbnAubmRhcnJheToKICAgIHBsID0gbG0ucHJlZGljdF9wcm9iYShYKVs6LCAxXS5hc3R5cGUobnAuZmxvYXQzMikKICAgIHB4ID0geG0ucHJlZGljdF9wcm9iYShYKVs6LCAxXS5hc3R5cGUobnAuZmxvYXQzMikKICAgIGlmIHByb2IgPT0gImxnYiI6CiAgICAgICAgcmV0dXJuIHBsCiAgICBpZiBwcm9iID09ICJ4Z2IiOgogICAgICAgIHJldHVybiBweAogICAgcmV0dXJuIDAuNSAqIChwbCArIHB4KQoKCmRlZiBfaXRlcl9jb3VudHJ5X3MxKGNvdW50cnk6IHN0ciwgbGltaXQ6IGludCB8IE5vbmUpOgogICAgIiIiWWllbGQgKHJpZCwgY29yZSwgYWRkcikgZm9yIHRlc3QgUzEgcm93cyBvZiBvbmUgY291bnRyeSAoc3RyZWFtZWQpLiIiIgogICAgc2VlbiA9IDAKICAgIGZvciBjaHVuayBpbiBjLml0ZXJfY2xlYW4oYy5URVNUX0ZJTEVTWyJTMSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICB1c2Vjb2xzPVsiZW50aXR5X2lkIiwgImNvdW50cnkiLCAibmFtZV9jb3JlIiwgImFkZHJfbm9ybSJdKToKICAgICAgICBzdWIgPSBjaHVua1tjaHVua1siY291bnRyeSJdLnZhbHVlcyA9PSBjb3VudHJ5XQogICAgICAgIGZvciByaWQsIGNvcmUsIGFkZHIgaW4gemlwKHN1YlsiZW50aXR5X2lkIl0sIHN1YlsibmFtZV9jb3JlIl0sIHN1YlsiYWRkcl9ub3JtIl0pOgogICAgICAgICAgICB5aWVsZCByaWQsIGNvcmUsIGFkZHIKICAgICAgICAgICAgc2VlbiArPSAxCiAgICAgICAgICAgIGlmIGxpbWl0IGFuZCBzZWVuID49IGxpbWl0OgogICAgICAgICAgICAgICAgcmV0dXJuCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRvcGsiLCB0eXBlPWludCwgZGVmYXVsdD0yMDApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtczEiLCB0eXBlPWludCwgZGVmYXVsdD00MDAwLAogICAgICAgICAgICAgICAgICAgIGhlbHA9IlMxIGVudGl0aWVzIHBlciBmZWF0dXJlL3ByZWRpY3QgYmF0Y2ggKG1lbW9yeSBrbm9iKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvY3MiLCB0eXBlPWludCwgZGVmYXVsdD0xMCwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJmZWF0dXJlLWNvbXB1dGUgd29ya2VyIHByb2Nlc3NlcyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGltaXQiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImNhcCBTMSBQRVIgQ09VTlRSWSAoc21va2UgdGVzdCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvdW50cmllcyIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJjb21tYSBsaXN0IHRvIHJlc3RyaWN0IChkZWZhdWx0OiBhbGwgaW4gdGVzdCBTMSkiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGltcG9ydCBsaWdodGdibSAgIyBub3FhOiBGNDAxICAoaW1wb3J0ZWQgaGVyZSBzbyB3b3JrZXJzIGRvbid0IHBheSBmb3IgaXQpCiAgICBpbXBvcnQgeGdib29zdCAgICMgbm9xYTogRjQwMQoKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oYy5NT0RFTFMsICJkZXBsb3lfbGdiLnBrbCIpLCAicmIiKSBhcyBmOgogICAgICAgIGxtID0gcGlja2xlLmxvYWQoZikKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oYy5NT0RFTFMsICJkZXBsb3lfeGdiLnBrbCIpLCAicmIiKSBhcyBmOgogICAgICAgIHhtID0gcGlja2xlLmxvYWQoZikKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oYy5NT0RFTFMsICJkZXBsb3lfbWV0YS5qc29uIikpIGFzIGY6CiAgICAgICAgbWV0YSA9IGpzb24ubG9hZChmKQogICAgdF9zdGFyID0gZmxvYXQobWV0YVsidGhyZXNob2xkIl0pCiAgICBwcm9iID0gbWV0YS5nZXQoInByb2IiLCAiYmxlbmQiKQogICAgYXNzZXJ0IG1ldGFbImZlYXR1cmVzIl0gPT0gRi5GRUFUVVJFUywgImRlcGxveSBmZWF0dXJlIG9yZGVyICE9IGZlYXR1cmVzLkZFQVRVUkVTIgogICAgcHJpbnQoZiJbaW5mZXJdIG1vZGVscyBsb2FkZWQgIHQqPXt0X3N0YXI6LjNmfSAgcHJvYj17cHJvYn0gICIKICAgICAgICAgIGYidG9waz17YXJncy50b3BrfSBiYXRjaD17YXJncy5iYXRjaF9zMX0gcHJvY3M9e2FyZ3MucHJvY3N9IiwgZmx1c2g9VHJ1ZSkKCiAgICBjb3VudHJpZXMgPSAoYXJncy5jb3VudHJpZXMuc3BsaXQoIiwiKSBpZiBhcmdzLmNvdW50cmllcwogICAgICAgICAgICAgICAgIGVsc2UgYmwuZGlzdGluY3RfY291bnRyaWVzKGMuVEVTVF9GSUxFUykpCiAgICBwcmludChmIltpbmZlcl0gY291bnRyaWVzPXtjb3VudHJpZXN9IiwgZmx1c2g9VHJ1ZSkKCiAgICBvcy5tYWtlZGlycyhjLk9VVFBVVCwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG1hdGNoX3BhdGggPSBvcy5wYXRoLmpvaW4oYy5PVVRQVVQsICJtYXRjaGluZ19yZXN1bHRzLnRzdiIpCiAgICBjYW5kX3BhdGggPSBvcy5wYXRoLmpvaW4oYy5PVVRQVVQsICJjYW5kaWRhdGVfcGFpcnMudHN2IikKICAgIHRfYWxsID0gdGltZS50aW1lKCkKICAgIG5fczEgPSBuX3BhaXJzID0gbl9wcmVkX3BhaXJzID0gbl9ub25lbXB0eSA9IDAKCiAgICBwb29sID0gUG9vbChhcmdzLnByb2NzLCBpbml0aWFsaXplcj1fd2luaXQsIGluaXRhcmdzPShJREZfUEFUSCwgTlVNSURGX1BBVEgpKQogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbihtYXRjaF9wYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgbWYsIFwKICAgICAgICAgICAgIG9wZW4oY2FuZF9wYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgY2Y6CiAgICAgICAgICAgIG1mLndyaXRlKCJzb3VyY2UxX2VudGl0eV9pZFx0bWF0Y2hlZF9lbnRpdHlfaWRzXG4iKQogICAgICAgICAgICBjZi53cml0ZSgic291cmNlMV9lbnRpdHlfaWRcdGNhbmRpZGF0ZV9lbnRpdHlfaWRzXG4iKQoKICAgICAgICAgICAgZm9yIGNvdW50cnkgaW4gY291bnRyaWVzOgogICAgICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgY2kgPSBibC5sb2FkX2NvdW50cnlfaW5kZXgoYy5URVNUX0ZJTEVTLCBjb3VudHJ5LCBkZl9jYXA9QlVJTERfQ0FQKQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIFt7Y291bnRyeX1dIGluZGV4OiB7Y2kubl9yZWNvcmRzKCk6LH0gcmVjcyBpbiAiCiAgICAgICAgICAgICAgICAgICAgICBmInt0aW1lLnRpbWUoKS10MDouMGZ9cyIsIGZsdXNoPVRydWUpCgogICAgICAgICAgICAgICAgcTAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgY25fczEgPSBjbl9wYWlycyA9IGNuX3ByZWQgPSAwCiAgICAgICAgICAgICAgICBiYXRjaCA9IFtdICAgICAgICAgICMgKHJpZCwgY29yZSwgYWRkcikKCiAgICAgICAgICAgICAgICBkZWYgZmx1c2goYmF0Y2gpOgogICAgICAgICAgICAgICAgICAgIG5vbmxvY2FsIGNuX3MxLCBjbl9wYWlycywgY25fcHJlZCwgbl9ub25lbXB0eQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBiYXRjaDoKICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgICAgICAgICAgcmlkcywgY2lkc19wZXIsIHJvd3MgPSBbXSwgW10sIFtdCiAgICAgICAgICAgICAgICAgICAgZm9yIHJpZCwgY29yZSwgYWRkciBpbiBiYXRjaDoKICAgICAgICAgICAgICAgICAgICAgICAgcG9zID0gYmwuY2FuZGlkYXRlc19mb3IoY2ksIGNvcmUsIGFkZHIsIGFyZ3MudG9waywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZWZmX2NhcD1CVUlMRF9DQVAsIGRldGFpbF9jYXA9MCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuX3Bvcz1UcnVlKQogICAgICAgICAgICAgICAgICAgICAgICBjaWRzID0gW2NpLmlkc1tqXSBmb3IgaiBpbiBwb3NdCiAgICAgICAgICAgICAgICAgICAgICAgIHJpZHMuYXBwZW5kKHJpZCkKICAgICAgICAgICAgICAgICAgICAgICAgY2lkc19wZXIuYXBwZW5kKGNpZHMpCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBqLCBjaWQgaW4gemlwKHBvcywgY2lkcyk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCgoY2lkLCBjb3JlLCBhZGRyLCBjaS5jb3Jlc1tqXSwgY2kuYWRkcnNbal0pKQogICAgICAgICAgICAgICAgICAgIGNuX3MxICs9IGxlbihyaWRzKQoKICAgICAgICAgICAgICAgICAgICBpZiByb3dzOgogICAgICAgICAgICAgICAgICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZShyb3dzLCBjb2x1bW5zPVsiY2FuZF9pZCIsICJzMV9jb3JlIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInMxX2FkZHIiLCAiY19jb3JlIiwgImNfYWRkciJdKQogICAgICAgICAgICAgICAgICAgICAgICBjbl9wYWlycyArPSBsZW4oZGYpCiAgICAgICAgICAgICAgICAgICAgICAgIG5zcCA9IG1heCgxLCBsZW4oZGYpIC8vIGFyZ3MucHJvY3MpCiAgICAgICAgICAgICAgICAgICAgICAgIGNodW5rcyA9IFtkZi5pbG9jW2k6aSArIG5zcF0gZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKGRmKSwgbnNwKV0KICAgICAgICAgICAgICAgICAgICAgICAgcGFydHMgPSBwb29sLm1hcChfd2ZlYXQsIGNodW5rcykKICAgICAgICAgICAgICAgICAgICAgICAgWCA9IG5wLmNvbmNhdGVuYXRlKHBhcnRzKSBpZiBsZW4ocGFydHMpID4gMSBlbHNlIHBhcnRzWzBdCiAgICAgICAgICAgICAgICAgICAgICAgIGJsZW5kID0gX3ByZWRpY3RfYmxlbmQobG0sIHhtLCBYLCBwcm9iKQogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIGJsZW5kID0gbnAuZW1wdHkoMCwgZHR5cGU9bnAuZmxvYXQzMikKCiAgICAgICAgICAgICAgICAgICAgb2ZmID0gMAogICAgICAgICAgICAgICAgICAgIGZvciByaWQsIGNpZHMgaW4gemlwKHJpZHMsIGNpZHNfcGVyKToKICAgICAgICAgICAgICAgICAgICAgICAgayA9IGxlbihjaWRzKQogICAgICAgICAgICAgICAgICAgICAgICBwID0gYmxlbmRbb2ZmOm9mZiArIGtdCiAgICAgICAgICAgICAgICAgICAgICAgIG9mZiArPSBrCiAgICAgICAgICAgICAgICAgICAgICAgIGtlcHQgPSBbY2lkIGZvciBjaWQsIHBwIGluIHppcChjaWRzLCBwKSBpZiBwcCA+PSB0X3N0YXJdCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGtlcHQ6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBuX25vbmVtcHR5ICs9IDEKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNuX3ByZWQgKz0gbGVuKGtlcHQpCiAgICAgICAgICAgICAgICAgICAgICAgIG1mLndyaXRlKGYie3JpZH1cdHsnLCcuam9pbihrZXB0KX1cbiIpCiAgICAgICAgICAgICAgICAgICAgICAgIGNmLndyaXRlKGYie3JpZH1cdHsnLCcuam9pbihjaWRzKX1cbiIpCgogICAgICAgICAgICAgICAgZm9yIHJpZCwgY29yZSwgYWRkciBpbiBfaXRlcl9jb3VudHJ5X3MxKGNvdW50cnksIGFyZ3MubGltaXQpOgogICAgICAgICAgICAgICAgICAgIGJhdGNoLmFwcGVuZCgocmlkLCBjb3JlLCBhZGRyKSkKICAgICAgICAgICAgICAgICAgICBpZiBsZW4oYmF0Y2gpID49IGFyZ3MuYmF0Y2hfczE6CiAgICAgICAgICAgICAgICAgICAgICAgIGZsdXNoKGJhdGNoKQogICAgICAgICAgICAgICAgICAgICAgICBiYXRjaCA9IFtdCiAgICAgICAgICAgICAgICAgICAgICAgIGVsID0gdGltZS50aW1lKCkgLSBxMAogICAgICAgICAgICAgICAgICAgICAgICBwcmludChmIiAgICBbe2NvdW50cnl9XSB7Y25fczE6LH0gUzEgfCB7Y25fcGFpcnM6LH0gcGFpcnMgfCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYie2NuX3ByZWQ6LH0gbWF0Y2hlZCB8IHtjbl9zMS9lbDosLjBmfSBTMS9zIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgICAgIGZsdXNoKGJhdGNoKQoKICAgICAgICAgICAgICAgIG5fczEgKz0gY25fczEKICAgICAgICAgICAgICAgIG5fcGFpcnMgKz0gY25fcGFpcnMKICAgICAgICAgICAgICAgIG5fcHJlZF9wYWlycyArPSBjbl9wcmVkCiAgICAgICAgICAgICAgICBkZWwgY2kKICAgICAgICAgICAgICAgIHByaW50KGYiICBbe2NvdW50cnl9XSBET05FIHtjbl9zMTosfSBTMSwge2NuX3BhaXJzOix9IHBhaXJzLCAiCiAgICAgICAgICAgICAgICAgICAgICBmIntjbl9wcmVkOix9IHByZWRpY3RlZCBtYXRjaGVzIGluIHt0aW1lLnRpbWUoKS1xMDouMGZ9cyIsIGZsdXNoPVRydWUpCiAgICBmaW5hbGx5OgogICAgICAgIHBvb2wuY2xvc2UoKQogICAgICAgIHBvb2wuam9pbigpCgogICAgcHJpbnQoZiJbaW5mZXJdIHtuX3MxOix9IFMxIHwge25fcGFpcnM6LH0gc2NvcmVkIHBhaXJzIHwge25fcHJlZF9wYWlyczosfSBtYXRjaGVzICIKICAgICAgICAgIGYifCB7bl9ub25lbXB0eTosfSBub24tZW1wdHkgUzEgfCB7dGltZS50aW1lKCktdF9hbGw6LjBmfXMiLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoZiJbaW5mZXJdIC0+IHttYXRjaF9wYXRofSIsIGZsdXNoPVRydWUpCiAgICBwcmludChmIltpbmZlcl0gLT4ge2NhbmRfcGF0aH0iLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "preprocessing.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQcmVwcm9jZXNzaW5nIC8gZGF0YSBjbGVhbmluZyBmb3IgdGhlIEJ1c2luZXNzIEVudGl0eSBSZXNvbHV0aW9uIGNoYWxsZW5nZS4KClN0cmVhbWluZyAoY2h1bmtlZCkgY2xlYW5lciBvdmVyIGFsbCA2IHNvdXJjZSBmaWxlcy4gRm9yIGV2ZXJ5IHJvdyBpdCBwcm9kdWNlcwpub3JtYWxpemVkLCB0cmFuc2xpdGVyYXRlZCwgY29tcGFyaXNvbi1yZWFkeSBmaWVsZHMgV0lUSE9VVCBkaXNjYXJkaW5nIHRoZSByYXcKdGV4dCwgYW5kIHdyaXRlcyBlbnJpY2hlZCBUU1ZzIHRvIGRhdGFzZXQvY2xlYW4vLiAgT2ZmbGluZSBvbmx5ICh1bmlkZWNvZGUncwpzdGF0aWMgdGFibGUgKyByZWdleCk7IG5vIGV4dGVybmFsIGRhdGEgbG9va3VwLCBwZXIgY2hhbGxlbmdlIHJ1bGVzLgoKT3V0cHV0IGNvbHVtbnM6CiAgICBlbnRpdHlfaWQsIGNvdW50cnksIGJ1c2luZXNzX25hbWUsIGJ1c2luZXNzX2FkZHJlc3MsCiAgICBuYW1lX25vcm0sIG5hbWVfY29yZSwgbmFtZV9zdWZmaXgsIGFkZHJfbm9ybQoKUnVuOiAgcHl0aG9uIGNvZGUvYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24vc3JjL3ByZXByb2Nlc3NpbmcucHkKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKaW1wb3J0IHBhbmRhcyBhcyBwZAp0cnk6CiAgICBmcm9tIHVuaWRlY29kZSBpbXBvcnQgdW5pZGVjb2RlCmV4Y2VwdCBFeGNlcHRpb246CiAgICB0cnk6CiAgICAgICAgZnJvbSBhbnlhc2NpaSBpbXBvcnQgYW55YXNjaWkgYXMgdW5pZGVjb2RlCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGltcG9ydCB1bmljb2RlZGF0YQogICAgICAgIGRlZiB1bmlkZWNvZGUocyk6CiAgICAgICAgICAgIHJldHVybiB1bmljb2RlZGF0YS5ub3JtYWxpemUoJ05GS0QnLCBzKS5lbmNvZGUoJ2FzY2lpJywgJ2lnbm9yZScpLmRlY29kZSgnYXNjaWknKQoKdHJ5OgogICAgc3lzLnN0ZG91dC5yZWNvbmZpZ3VyZShlbmNvZGluZz0idXRmLTgiLCBlcnJvcnM9InJlcGxhY2UiKQogICAgc3lzLnN0ZGVyci5yZWNvbmZpZ3VyZShlbmNvZGluZz0idXRmLTgiLCBlcnJvcnM9InJlcGxhY2UiKQpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwoKSEVSRSA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKQpST09UID0gb3MucGF0aC5hYnNwYXRoKG9zLnBhdGguam9pbihIRVJFLCAiLi4iLCAiLi4iLCAiLi4iKSkKT1VUX0RJUiA9IG9zLnBhdGguam9pbihST09ULCAiZGF0YXNldCIsICJjbGVhbiIpCgpDSFVOSyA9IDIwMF8wMDAKTk9OX0FTQ0lJID0gcmUuY29tcGlsZShyIlteXHgwMC1ceDdGXSIpCgpTT1VSQ0VTID0gWwogICAgKCJkYXRhc2V0L3RyYWluL3RyYWluX3NvdXJjZTEudHN2IiwgImRhdGFzZXQvY2xlYW4vdHJhaW4vdHJhaW5fc291cmNlMS50c3YiKSwKICAgICgiZGF0YXNldC90cmFpbi90cmFpbl9zb3VyY2UyLnRzdiIsICJkYXRhc2V0L2NsZWFuL3RyYWluL3RyYWluX3NvdXJjZTIudHN2IiksCiAgICAoImRhdGFzZXQvdHJhaW4vdHJhaW5fc291cmNlMy50c3YiLCAiZGF0YXNldC9jbGVhbi90cmFpbi90cmFpbl9zb3VyY2UzLnRzdiIpLAogICAgKCJkYXRhc2V0L3Rlc3QvdGVzdF9zb3VyY2UxLnRzdiIsICJkYXRhc2V0L2NsZWFuL3Rlc3QvdGVzdF9zb3VyY2UxLnRzdiIpLAogICAgKCJkYXRhc2V0L3Rlc3QvdGVzdF9zb3VyY2UyLnRzdiIsICJkYXRhc2V0L2NsZWFuL3Rlc3QvdGVzdF9zb3VyY2UyLnRzdiIpLAogICAgKCJkYXRhc2V0L3Rlc3QvdGVzdF9zb3VyY2UzLnRzdiIsICJkYXRhc2V0L2NsZWFuL3Rlc3QvdGVzdF9zb3VyY2UzLnRzdiIpLApdCgojIC0tLSBMZWdhbCBzdWZmaXhlczogdmFyaWFudCAtPiBjYW5vbmljYWwgdG9rZW4gKG9yZGVyLWFnbm9zdGljLCBkb3RsZXNzKSAtLS0tLQpMRUdBTF9TVUZGSVhfQ0FOT04gPSB7CiAgICAibGxjIjogImxsYyIsICJsLmwuYyI6ICJsbGMiLCAibGxjLiI6ICJsbGMiLAogICAgImluYyI6ICJpbmMiLCAiaW5jb3Jwb3JhdGVkIjogImluYyIsICJpbmMuIjogImluYyIsCiAgICAiY29ycCI6ICJjb3JwIiwgImNvcnBvcmF0aW9uIjogImNvcnAiLCAiY29ycC4iOiAiY29ycCIsCiAgICAiY28iOiAiY28iLCAiY29tcGFueSI6ICJjbyIsICJjby4iOiAiY28iLAogICAgImx0ZCI6ICJsdGQiLCAibGltaXRlZCI6ICJsdGQiLCAibHRkLiI6ICJsdGQiLAogICAgInB2dCI6ICJwdnQiLCAicHJpdmF0ZSI6ICJwdnQiLCAicHZ0LiI6ICJwdnQiLAogICAgImxscCI6ICJsbHAiLCAibC5sLnAiOiAibGxwIiwKICAgICJwbGMiOiAicGxjIiwgInBsbGMiOiAicGxsYyIsICJwYyI6ICJwYyIsICJwYSI6ICJwYSIsCiAgICAibHAiOiAibHAiLCAib3BjIjogIm9wYyIsICJuaWRoaSI6ICJuaWRoaSIsCiAgICAjIEZyZW5jaAogICAgInNhcmwiOiAic2FybCIsICJzYXN1IjogInNhc3UiLCAic2FzIjogInNhcyIsICJzYSI6ICJzYSIsCiAgICAiZXVybCI6ICJldXJsIiwgInNjaSI6ICJzY2kiLCAic25jIjogInNuYyIsICJzY2EiOiAic2NhIiwKICAgICJzYXJsdSI6ICJzYXJsIiwgImFzc29jaWF0aW9uIjogImFzc29jaWF0aW9uIiwgImdpZSI6ICJnaWUiLAp9CiMgTXVsdGktd29yZCBwaHJhc2VzIGNvbGxhcHNlZCBiZWZvcmUgdG9rZW4gc2Nhbi4KU1VGRklYX1BIUkFTRVMgPSBbCiAgICAociJcYnByaXZhdGVccytsaW1pdGVkXGIiLCAicHZ0IGx0ZCIpLAogICAgKHIiXGJwdnRcLj9ccypsdGRcLj9cYiIsICJwdnQgbHRkIiksCiAgICAociJcYnB1YmxpY1xzK2xpbWl0ZWRcYiIsICJsdGQiKSwKICAgIChyIlxibGltaXRlZFxzK2xpYWJpbGl0eVxzK2NvbXBhbnlcYiIsICJsbGMiKSwKICAgIChyIlxib25lXHMrcGVyc29uXHMrY29tcGFueVxiIiwgIm9wYyIpLApdClNVRkZJWF9UT0tFTlMgPSBzZXQoTEVHQUxfU1VGRklYX0NBTk9OLnZhbHVlcygpKSB8IHNldChMRUdBTF9TVUZGSVhfQ0FOT04ua2V5cygpKQoKIyAtLS0gU3RyZWV0LXR5cGUgYWJicmV2aWF0aW9ucyAoVVMgKyBhIGZldyBGUikgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KU1RSRUVUX0FCQlIgPSB7CiAgICAic3QiOiAic3RyZWV0IiwgInN0LiI6ICJzdHJlZXQiLCAic3RyIjogInN0cmVldCIsCiAgICAicmQiOiAicm9hZCIsICJyZC4iOiAicm9hZCIsICJhdmUiOiAiYXZlbnVlIiwgImF2ZS4iOiAiYXZlbnVlIiwKICAgICJhdiI6ICJhdmVudWUiLCAiYmx2ZCI6ICJib3VsZXZhcmQiLCAiYmx2ZC4iOiAiYm91bGV2YXJkIiwgImJkIjogImJvdWxldmFyZCIsCiAgICAiZHIiOiAiZHJpdmUiLCAiZHIuIjogImRyaXZlIiwgImxuIjogImxhbmUiLCAiY3QiOiAiY291cnQiLAogICAgInBsIjogInBsYWNlIiwgImNpciI6ICJjaXJjbGUiLCAicGt3eSI6ICJwYXJrd2F5IiwgImh3eSI6ICJoaWdod2F5IiwKICAgICJzcSI6ICJzcXVhcmUiLCAidGVyIjogInRlcnJhY2UiLCAidHJsIjogInRyYWlsIiwgImV4cHkiOiAiZXhwcmVzc3dheSIsCiAgICAiaG5vIjogImhvdXNlIG5vIiwgImgubm8iOiAiaG91c2Ugbm8iLAp9CgojIC0tLSBTdGF0ZSAvIHJlZ2lvbjogY29kZSAtPiBjYW5vbmljYWwgZnVsbCBuYW1lLCBrZXB0IHBlci1jb3VudHJ5IHRvIGF2b2lkCiMgICAgIHRoZSBVUy9JbmRpYSAyLWxldHRlciBjb2xsaXNpb25zIChjYSwgY28sIGluLCBtcCwgb3IsIC4uLikuIEFwcGxpZWQgb25seQojICAgICB0byBhIGNvbW1hIHNlZ21lbnQgdGhhdCBpcyBFWEFDVExZIGEgdmFsaWQgY29kZSBmb3IgdGhlIHJvdydzIGNvdW50cnksIHNvCiMgICAgIHdlIG5ldmVyIHJld3JpdGUgYSByZWFsIHdvcmQuIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KVVNfU1RBVEUgPSB7CiAgICAiYWwiOiAiYWxhYmFtYSIsICJhayI6ICJhbGFza2EiLCAiYXoiOiAiYXJpem9uYSIsICJhciI6ICJhcmthbnNhcyIsCiAgICAiY2EiOiAiY2FsaWZvcm5pYSIsICJjbyI6ICJjb2xvcmFkbyIsICJjdCI6ICJjb25uZWN0aWN1dCIsICJkZSI6ICJkZWxhd2FyZSIsCiAgICAiZmwiOiAiZmxvcmlkYSIsICJnYSI6ICJnZW9yZ2lhIiwgImhpIjogImhhd2FpaSIsICJpZCI6ICJpZGFobyIsCiAgICAiaWwiOiAiaWxsaW5vaXMiLCAiaW4iOiAiaW5kaWFuYSIsICJpYSI6ICJpb3dhIiwgImtzIjogImthbnNhcyIsCiAgICAia3kiOiAia2VudHVja3kiLCAibGEiOiAibG91aXNpYW5hIiwgIm1lIjogIm1haW5lIiwgIm1kIjogIm1hcnlsYW5kIiwKICAgICJtYSI6ICJtYXNzYWNodXNldHRzIiwgIm1pIjogIm1pY2hpZ2FuIiwgIm1uIjogIm1pbm5lc290YSIsICJtcyI6ICJtaXNzaXNzaXBwaSIsCiAgICAibW8iOiAibWlzc291cmkiLCAibXQiOiAibW9udGFuYSIsICJuZSI6ICJuZWJyYXNrYSIsICJudiI6ICJuZXZhZGEiLAogICAgIm5oIjogIm5ldyBoYW1wc2hpcmUiLCAibmoiOiAibmV3IGplcnNleSIsICJubSI6ICJuZXcgbWV4aWNvIiwgIm55IjogIm5ldyB5b3JrIiwKICAgICJuYyI6ICJub3J0aCBjYXJvbGluYSIsICJuZCI6ICJub3J0aCBkYWtvdGEiLCAib2giOiAib2hpbyIsICJvayI6ICJva2xhaG9tYSIsCiAgICAib3IiOiAib3JlZ29uIiwgInBhIjogInBlbm5zeWx2YW5pYSIsICJyaSI6ICJyaG9kZSBpc2xhbmQiLCAic2MiOiAic291dGggY2Fyb2xpbmEiLAogICAgInNkIjogInNvdXRoIGRha290YSIsICJ0biI6ICJ0ZW5uZXNzZWUiLCAidHgiOiAidGV4YXMiLCAidXQiOiAidXRhaCIsCiAgICAidnQiOiAidmVybW9udCIsICJ2YSI6ICJ2aXJnaW5pYSIsICJ3YSI6ICJ3YXNoaW5ndG9uIiwgInd2IjogIndlc3QgdmlyZ2luaWEiLAogICAgIndpIjogIndpc2NvbnNpbiIsICJ3eSI6ICJ3eW9taW5nIiwgImRjIjogImRpc3RyaWN0IG9mIGNvbHVtYmlhIiwKfQpJTl9TVEFURSA9IHsKICAgICJhcCI6ICJhbmRocmEgcHJhZGVzaCIsICJhcyI6ICJhc3NhbSIsICJiciI6ICJiaWhhciIsICJjZyI6ICJjaGhhdHRpc2dhcmgiLAogICAgImdhIjogImdvYSIsICJnaiI6ICJndWphcmF0IiwgImhyIjogImhhcnlhbmEiLCAiaHAiOiAiaGltYWNoYWwgcHJhZGVzaCIsCiAgICAiamgiOiAiamhhcmtoYW5kIiwgImthIjogImthcm5hdGFrYSIsICJrbCI6ICJrZXJhbGEiLCAibXAiOiAibWFkaHlhIHByYWRlc2giLAogICAgIm1oIjogIm1haGFyYXNodHJhIiwgIm1sIjogIm1lZ2hhbGF5YSIsICJteiI6ICJtaXpvcmFtIiwgIm5sIjogIm5hZ2FsYW5kIiwKICAgICJvZCI6ICJvZGlzaGEiLCAicGIiOiAicHVuamFiIiwgInJqIjogInJhamFzdGhhbiIsICJzayI6ICJzaWtraW0iLAogICAgInRuIjogInRhbWlsIG5hZHUiLCAidGciOiAidGVsYW5nYW5hIiwgInRzIjogInRlbGFuZ2FuYSIsICJ0ciI6ICJ0cmlwdXJhIiwKICAgICJ1cCI6ICJ1dHRhciBwcmFkZXNoIiwgInVrIjogInV0dGFyYWtoYW5kIiwgIndiIjogIndlc3QgYmVuZ2FsIiwgImRsIjogImRlbGhpIiwKICAgICJqayI6ICJqYW1tdSBhbmQga2FzaG1pciIsCn0KU1RBVEVfQllfQ09VTlRSWSA9IHsiVVMiOiBVU19TVEFURSwgIkluZGlhIjogSU5fU1RBVEV9CgojIExhbmRtYXJrIC8gbm9pc2UgcGhyYXNlcyBzdHJpcHBlZCBmcm9tIGFkZHJlc3NlcyAoZG8gbm90IGhlbHAgbWF0Y2hpbmcpLgpMQU5ETUFSS19SRSA9IHJlLmNvbXBpbGUoCiAgICByIlxiKG5lYXJ8bnJ8b3BwfG9wcG9zaXRlfGJlaGluZHxiZXNpZGV8bmV4dCB0b3xhZGphY2VudCB0b3xhYm92ZXxiZWxvdylcYlteLF0qIiwKICAgIHJlLklHTk9SRUNBU0UsCikKCiMgLS0tIFJlZ2V4IGJ1aWxkaW5nIGJsb2NrcyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU3RyaXAgcHJvdG9jb2wvd3d3IGFuZCB0aGUgdHJhaWxpbmcgVExEIG9ubHksIHNvICJoZWFzc29jaWF0ZXMuY29tIiBrZWVwcyB0aGUKIyBtYXRjaGFibGUgdG9rZW4gImhlYXNzb2NpYXRlcyIgaW5zdGVhZCBvZiB2YW5pc2hpbmcuCldFQl9QUkVGSVhfUkUgPSByZS5jb21waWxlKHIiXGIoPzpodHRwcz86Ly98d3d3XC4pIikKVExEX1JFID0gcmUuY29tcGlsZShyIlwuKD86Y29tfG5ldHxvcmd8aW58Y298aW98ZnJ8Yml6fGluZm8pXGIiKQpOVUxMX1JFID0gcmUuY29tcGlsZShyIjw/XGJudWxsXGI+PyIsIHJlLklHTk9SRUNBU0UpCkpVTktfQ0hBUlNfUkUgPSByZS5jb21waWxlKHIiW1xbXF17fSgpPD4jKlwifH5eYF9cXF0rIikKREFTSF9SVU5fUkUgPSByZS5jb21waWxlKHIiKD86XnxccylbLeKAk+KAlF17MSx9KD89XHN8JCkiKQpQSE9ORV9SRSA9IHJlLmNvbXBpbGUociJcYlxkezcsfVxiIikKTVVMVElTUEFDRV9SRSA9IHJlLmNvbXBpbGUociJccysiKQpUT0tFTl9SRSA9IHJlLmNvbXBpbGUociJbYS16MC05XSsiKQpOT05fQUxOVU1fU1BBQ0VfUkUgPSByZS5jb21waWxlKHIiW15hLXowLTkgXSsiKQoKCmRlZiBfdHJhbnNsaXQoczogc3RyKSAtPiBzdHI6CiAgICAiIiJSb21hbml6ZSBvbmx5IHdoZW4gbmVlZGVkIChub24tQVNDSUkpOyB1bmlkZWNvZGUgaXMgdGhlIG9mZmxpbmUgdGFibGUuIiIiCiAgICBpZiBzIGFuZCBOT05fQVNDSUkuc2VhcmNoKHMpOgogICAgICAgIHJldHVybiB1bmlkZWNvZGUocykKICAgIHJldHVybiBzCgoKZGVmIG5vcm1hbGl6ZV9uYW1lKHJhdzogc3RyKSAtPiB0dXBsZVtzdHIsIHN0ciwgc3RyXToKICAgICIiInJhdyBidXNpbmVzc19uYW1lIC0+IChuYW1lX25vcm0sIG5hbWVfY29yZSwgbmFtZV9zdWZmaXgpLgoKICAgIG5hbWVfbm9ybSAgIDogdHJhbnNsaXRlcmF0ZWQsIGxvd2VyY2FzZWQsIGRlLWp1bmtlZCwgJyYnLT4nYW5kJywgd3MtY29sbGFwc2VkCiAgICBuYW1lX3N1ZmZpeCA6IHNvcnRlZCBjYW5vbmljYWwgbGVnYWwtc3VmZml4IHRva2VucyBmb3VuZCAoc3BhY2Utam9pbmVkKQogICAgbmFtZV9jb3JlICAgOiBuYW1lX25vcm0gd2l0aCB0aG9zZSBzdWZmaXggdG9rZW5zIHJlbW92ZWQgKGZvciBuYW1lIGNvbXBhcmlzb24pCiAgICAiIiIKICAgIHMgPSBfdHJhbnNsaXQocmF3KS5sb3dlcigpCiAgICBzID0gV0VCX1BSRUZJWF9SRS5zdWIoIiAiLCBzKQogICAgcyA9IFRMRF9SRS5zdWIoIiAiLCBzKQogICAgcyA9IHMucmVwbGFjZSgiJiIsICIgYW5kICIpLnJlcGxhY2UoIisiLCAiIGFuZCAiKQogICAgcyA9IHJlLnN1YihyIlxiZFwuP2JcLj9hXC4/XGIiLCAiICIsIHMpICAgICAgICAgICMgZHJvcCBEQkEgbWFya2VyCiAgICBzID0gSlVOS19DSEFSU19SRS5zdWIoIiAiLCBzKQogICAgcyA9IFBIT05FX1JFLnN1YigiICIsIHMpCiAgICBmb3IgcGF0LCByZXBsIGluIFNVRkZJWF9QSFJBU0VTOgogICAgICAgIHMgPSByZS5zdWIocGF0LCByZXBsLCBzKQogICAgcyA9IE5PTl9BTE5VTV9TUEFDRV9SRS5zdWIoIiAiLCBzKQogICAgcyA9IE1VTFRJU1BBQ0VfUkUuc3ViKCIgIiwgcykuc3RyaXAoKQoKICAgIHRva3MgPSBzLnNwbGl0KCkKICAgIHN1ZmZpeGVzLCBjb3JlID0gW10sIFtdCiAgICBmb3IgdCBpbiB0b2tzOgogICAgICAgIGNhbm9uID0gTEVHQUxfU1VGRklYX0NBTk9OLmdldCh0KQogICAgICAgIGlmIGNhbm9uIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzdWZmaXhlcy5hcHBlbmQoY2Fub24pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY29yZS5hcHBlbmQodCkKICAgIG5hbWVfY29yZSA9ICIgIi5qb2luKGNvcmUpLnN0cmlwKCkgb3IgcyAgICAgIyBuZXZlciBsZWF2ZSBjb3JlIGVtcHR5CiAgICBuYW1lX3N1ZmZpeCA9ICIgIi5qb2luKHNvcnRlZChzZXQoc3VmZml4ZXMpKSkKICAgIHJldHVybiBzLCBuYW1lX2NvcmUsIG5hbWVfc3VmZml4CgoKZGVmIG5vcm1hbGl6ZV9hZGRyZXNzKHJhdzogc3RyLCBjb3VudHJ5OiBzdHIpIC0+IHN0cjoKICAgICIiIlRyYW5zbGl0ZXJhdGUgKyBkZS1ub2lzZSBhbiBhZGRyZXNzIGludG8gYSBjb21wYXJpc29uLXJlYWR5IHN0cmluZy4iIiIKICAgIGlmIG5vdCByYXc6CiAgICAgICAgcmV0dXJuICIiCiAgICBzID0gX3RyYW5zbGl0KHJhdykubG93ZXIoKQogICAgcyA9IE5VTExfUkUuc3ViKCIgIiwgcykKICAgIHMgPSBXRUJfUFJFRklYX1JFLnN1YigiICIsIHMpCiAgICBzID0gVExEX1JFLnN1YigiICIsIHMpCiAgICBzID0gTEFORE1BUktfUkUuc3ViKCIgIiwgcykKICAgIHMgPSBKVU5LX0NIQVJTX1JFLnN1YigiICIsIHMpCgogICAgc3RhdGVfbWFwID0gU1RBVEVfQllfQ09VTlRSWS5nZXQoY291bnRyeSkKICAgIG91dF9zZWdtZW50cyA9IFtdCiAgICBmb3Igc2VnIGluIHMuc3BsaXQoIiwiKToKICAgICAgICBzZWcgPSBzZWcuc3RyaXAoKQogICAgICAgIGlmIG5vdCBzZWc6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIyBhIHNlZ21lbnQgdGhhdCBpcyBleGFjdGx5IGEgc3RhdGUgY29kZSAtPiBleHBhbmQgdG8gZnVsbCBuYW1lCiAgICAgICAgaWYgc3RhdGVfbWFwIGFuZCBzZWcgaW4gc3RhdGVfbWFwOgogICAgICAgICAgICBvdXRfc2VnbWVudHMuYXBwZW5kKHN0YXRlX21hcFtzZWddKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRva3MgPSBbU1RSRUVUX0FCQlIuZ2V0KHQsIHQpIGZvciB0IGluIHNlZy5zcGxpdCgpXQogICAgICAgIG91dF9zZWdtZW50cy5hcHBlbmQoIiAiLmpvaW4odG9rcykpCiAgICBzID0gIiAiLmpvaW4ob3V0X3NlZ21lbnRzKQogICAgcyA9IHJlLnN1YihyIlteYS16MC05IF0rIiwgIiAiLCBzKSAgICAgICAgICAjIHRva2VucyBvbmx5OyBjb21tYXMgYWxyZWFkeSB1c2VkIGFib3ZlCiAgICBzID0gTVVMVElTUEFDRV9SRS5zdWIoIiAiLCBzKS5zdHJpcCgpCiAgICByZXR1cm4gcwoKCk9VVF9DT0xTID0gWwogICAgImVudGl0eV9pZCIsICJjb3VudHJ5IiwgImJ1c2luZXNzX25hbWUiLCAiYnVzaW5lc3NfYWRkcmVzcyIsCiAgICAibmFtZV9ub3JtIiwgIm5hbWVfY29yZSIsICJuYW1lX3N1ZmZpeCIsICJhZGRyX25vcm0iLApdCgoKZGVmIGNsZWFuX3NvdXJjZV9maWxlKGluX3JlbDogc3RyLCBvdXRfcmVsOiBzdHIpIC0+IGRpY3Q6CiAgICAiIiJTdHJlYW0gb25lIHNvdXJjZSBmaWxlLCBhZGQgbm9ybWFsaXplZCBjb2x1bW5zLCB3cml0ZSBlbnJpY2hlZCBUU1YuIiIiCiAgICBpbl9wYXRoID0gb3MucGF0aC5qb2luKFJPT1QsIGluX3JlbCkKICAgIG91dF9wYXRoID0gb3MucGF0aC5qb2luKFJPT1QsIG91dF9yZWwpCiAgICBvcy5tYWtlZGlycyhvcy5wYXRoLmRpcm5hbWUob3V0X3BhdGgpLCBleGlzdF9vaz1UcnVlKQoKICAgIHN0YXRzID0geyJyb3dzIjogMCwgInJlc2lkdWFsX25vbmFzY2lpX25hbWUiOiAwLCAiZW1wdHlfY29yZSI6IDAsCiAgICAgICAgICAgICAid2l0aF9zdWZmaXgiOiAwLCAiZW1wdHlfYWRkciI6IDB9CiAgICByZWFkZXIgPSBwZC5yZWFkX2NzdigKICAgICAgICBpbl9wYXRoLCBzZXA9Ilx0IiwgZHR5cGU9c3RyLCBrZWVwX2RlZmF1bHRfbmE9RmFsc2UsIG5hX3ZhbHVlcz1bXSwKICAgICAgICBjaHVua3NpemU9Q0hVTkssIHF1b3Rpbmc9MywgZW5jb2Rpbmc9InV0Zi04IiwgZW5jb2RpbmdfZXJyb3JzPSJyZXBsYWNlIiwKICAgICkKICAgIGZpcnN0ID0gVHJ1ZQogICAgZm9yIGNodW5rIGluIHJlYWRlcjoKICAgICAgICBjaHVuayA9IGNodW5rLmZpbGxuYSgiIikKICAgICAgICBuYW1lcyA9IGNodW5rWyJidXNpbmVzc19uYW1lIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgICAgICBhZGRycyA9IGNodW5rWyJidXNpbmVzc19hZGRyZXNzIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgICAgICBjdHJ5ID0gY2h1bmtbImNvdW50cnkiXS5hc3R5cGUoc3RyKS50b2xpc3QoKQoKICAgICAgICBubiA9IFtub3JtYWxpemVfbmFtZSh4KSBmb3IgeCBpbiBuYW1lc10KICAgICAgICBjaHVua1sibmFtZV9ub3JtIl0gPSBbdFswXSBmb3IgdCBpbiBubl0KICAgICAgICBjaHVua1sibmFtZV9jb3JlIl0gPSBbdFsxXSBmb3IgdCBpbiBubl0KICAgICAgICBjaHVua1sibmFtZV9zdWZmaXgiXSA9IFt0WzJdIGZvciB0IGluIG5uXQogICAgICAgIGNodW5rWyJhZGRyX25vcm0iXSA9IFtub3JtYWxpemVfYWRkcmVzcyhhLCBjKSBmb3IgYSwgYyBpbiB6aXAoYWRkcnMsIGN0cnkpXQoKICAgICAgICAjIGtlZXAgdGhlIHJhdyB0ZXh0IGJ1dCBzdHJpcCBzdHJheSB0YWIvbmV3bGluZSBzbyB0aGUgVFNWIHN0YXlzIDEgcm93L3JlY29yZAogICAgICAgIGNodW5rWyJidXNpbmVzc19uYW1lIl0gPSBjaHVua1siYnVzaW5lc3NfbmFtZSJdLnN0ci5yZXBsYWNlKHIiW1x0XHJcbl0rIiwgIiAiLCByZWdleD1UcnVlKQogICAgICAgIGNodW5rWyJidXNpbmVzc19hZGRyZXNzIl0gPSBjaHVua1siYnVzaW5lc3NfYWRkcmVzcyJdLnN0ci5yZXBsYWNlKHIiW1x0XHJcbl0rIiwgIiAiLCByZWdleD1UcnVlKQoKICAgICAgICBvdXQgPSBjaHVua1tPVVRfQ09MU10KICAgICAgICBvdXQudG9fY3N2KAogICAgICAgICAgICBvdXRfcGF0aCwgc2VwPSJcdCIsIGluZGV4PUZhbHNlLCBtb2RlPSJ3IiBpZiBmaXJzdCBlbHNlICJhIiwKICAgICAgICAgICAgaGVhZGVyPWZpcnN0LCBlbmNvZGluZz0idXRmLTgiLCBsaW5ldGVybWluYXRvcj0iXG4iLAogICAgICAgICkKICAgICAgICBmaXJzdCA9IEZhbHNlCgogICAgICAgIHN0YXRzWyJyb3dzIl0gKz0gbGVuKG91dCkKICAgICAgICBzdGF0c1sicmVzaWR1YWxfbm9uYXNjaWlfbmFtZSJdICs9IGludCgKICAgICAgICAgICAgb3V0WyJuYW1lX25vcm0iXS5zdHIuY29udGFpbnMoTk9OX0FTQ0lJKS5zdW0oKSkKICAgICAgICBzdGF0c1siZW1wdHlfY29yZSJdICs9IGludCgob3V0WyJuYW1lX2NvcmUiXS5zdHIubGVuKCkgPT0gMCkuc3VtKCkpCiAgICAgICAgc3RhdHNbIndpdGhfc3VmZml4Il0gKz0gaW50KChvdXRbIm5hbWVfc3VmZml4Il0uc3RyLmxlbigpID4gMCkuc3VtKCkpCiAgICAgICAgc3RhdHNbImVtcHR5X2FkZHIiXSArPSBpbnQoKG91dFsiYWRkcl9ub3JtIl0uc3RyLmxlbigpID09IDApLnN1bSgpKQogICAgICAgIHByaW50KGYiICAgIC4uLntzdGF0c1sncm93cyddOix9IHJvd3MiLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIHN0YXRzCgoKZGVmIHByZXZpZXcob3V0X3JlbDogc3RyLCBuOiBpbnQgPSA2KToKICAgIHBhdGggPSBvcy5wYXRoLmpvaW4oUk9PVCwgb3V0X3JlbCkKICAgIGRmID0gcGQucmVhZF9jc3YocGF0aCwgc2VwPSJcdCIsIGR0eXBlPXN0ciwga2VlcF9kZWZhdWx0X25hPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICBuYV92YWx1ZXM9W10sIG5yb3dzPW4sIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBmb3IgXywgciBpbiBkZi5pdGVycm93cygpOgogICAgICAgIHByaW50KGYiICAgIFJBVyAgbmFtZT17clsnYnVzaW5lc3NfbmFtZSddIXJ9ICBhZGRyPXtyWydidXNpbmVzc19hZGRyZXNzJ10hcn0gW3tyWydjb3VudHJ5J119XSIpCiAgICAgICAgcHJpbnQoZiIgICAgLT4gICBub3JtPXtyWyduYW1lX25vcm0nXSFyfSBjb3JlPXtyWyduYW1lX2NvcmUnXSFyfSAiCiAgICAgICAgICAgICAgZiJzdWY9e3JbJ25hbWVfc3VmZml4J10hcn0iKQogICAgICAgIHByaW50KGYiICAgIC0+ICAgYWRkcl9ub3JtPXtyWydhZGRyX25vcm0nXSFyfSIpCgoKZGVmIG1haW4oKToKICAgIHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoIlBSRVBST0NFU1NJTkcgLyBDTEVBTklORyDigJQgd3JpdGluZyBlbnJpY2hlZCBUU1ZzIHRvIGRhdGFzZXQvY2xlYW4vIikKICAgIHByaW50KCI9IiAqIDcwKQogICAgZ3JhbmQgPSAwCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBmb3IgaW5fcmVsLCBvdXRfcmVsIGluIFNPVVJDRVM6CiAgICAgICAgcHJpbnQoZiJcblt7aW5fcmVsfV0gLT4gW3tvdXRfcmVsfV0iKQogICAgICAgIHRzID0gdGltZS50aW1lKCkKICAgICAgICBzdCA9IGNsZWFuX3NvdXJjZV9maWxlKGluX3JlbCwgb3V0X3JlbCkKICAgICAgICBncmFuZCArPSBzdFsicm93cyJdCiAgICAgICAgbiA9IHN0WyJyb3dzIl0KICAgICAgICBwcmludChmIiAgZG9uZSB7bjosfSByb3dzIGluIHt0aW1lLnRpbWUoKS10czouMGZ9cyB8ICIKICAgICAgICAgICAgICBmInJlc2lkdWFsIG5vbi1BU0NJSSBuYW1lPXtzdFsncmVzaWR1YWxfbm9uYXNjaWlfbmFtZSddfSAiCiAgICAgICAgICAgICAgZiIoezEwMCpzdFsncmVzaWR1YWxfbm9uYXNjaWlfbmFtZSddL246LjNmfSUpIHwgIgogICAgICAgICAgICAgIGYiZW1wdHkgY29yZT17c3RbJ2VtcHR5X2NvcmUnXX0gfCB3aXRoIHN1ZmZpeD17c3RbJ3dpdGhfc3VmZml4J119ICIKICAgICAgICAgICAgICBmIih7MTAwKnN0Wyd3aXRoX3N1ZmZpeCddL246LjFmfSUpIHwgZW1wdHkgYWRkcl9ub3JtPXtzdFsnZW1wdHlfYWRkciddfSIpCiAgICAgICAgcHJldmlldyhvdXRfcmVsKQogICAgcHJpbnQoZiJcbkFMTCBET05FOiB7Z3JhbmQ6LH0gcm93cyBjbGVhbmVkIGluIHt0aW1lLnRpbWUoKS10MDouMGZ9cyIpCiAgICBwcmludChmIkNsZWFuZWQgZGF0YXNldCB3cml0dGVuIHVuZGVyOiB7T1VUX0RJUn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkKCgoKCgo=", "build_index.py": "aW1wb3J0IG9zLCBwaWNrbGUsIHRpbWUKaW1wb3J0IGNvbW1vbiBhcyBjLCBibG9ja2luZyBhcyBibApDQUNIRSA9IG9zLnBhdGguam9pbihjLldPUkssICdpZHhfaW5kaWFfY2FwMzAwMC5wa2wnKQp0MCA9IHRpbWUudGltZSgpCmNpID0gYmwubG9hZF9jb3VudHJ5X2luZGV4KGMuVFJBSU5fRklMRVMsICdJbmRpYScsIGRmX2NhcD0zMDAwKQp3aXRoIG9wZW4oQ0FDSEUsICd3YicpIGFzIGY6CiAgICBwaWNrbGUuZHVtcChjaSwgZiwgcHJvdG9jb2w9NCkKcHJpbnQoJ2luZGlhIGluZGV4ICVkIHJlY3MgY2FjaGVkIGluICVkcyAtPiAlcycgJSAoY2kubl9yZWNvcmRzKCksIHRpbWUudGltZSgpLXQwLCBDQUNIRSksIGZsdXNoPVRydWUpCg=="}""")
for name, b64 in FILES.items():
    with open(os.path.join(SRC_DIR, name), "wb") as f:
        f.write(base64.b64decode(b64))
print("wrote modules:", sorted(FILES))


In [ ]:
# --- subprocess runner (cwd=SRC_DIR so `import common` resolves; live logs) --
import os, sys, subprocess, time
ENV = dict(os.environ)
ENV["BER_OUTPUT"] = "/kaggle/working"     # the two deliverable TSVs land here (downloadable)
ENV["PYTHONUNBUFFERED"] = "1"

def run(script, *a):
    cmd = [sys.executable, os.path.join(SRC_DIR, script), *map(str, a)]
    print(">>", " ".join(cmd), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, cwd=SRC_DIR, env=ENV)
    if r.returncode != 0:
        raise RuntimeError(script + " FAILED rc=" + str(r.returncode))
    print("   [%s] done in %.1f min" % (script, (time.time() - t0) / 60), flush=True)


In [ ]:
# --- Stage 1: clean/normalize all six raw source files ----------------------
run("preprocessing.py")


In [ ]:
# --- Stage 2: India blocking index + IDFs + 20k labeled dev set + features ---
run("build_index.py")                 # -> work/idx_india_cap3000.pkl
run("make_devset.py", 20000)          # -> work/devset_india_n20000.parquet (+ tokdf_india.pkl)
run("build_numidf.py")                # -> work/numidf_india.pkl (rare-numeric address signal)
DEV = ROOT + "/dataset/work/devset_india_n20000.parquet"
run("features.py", DEV)               # -> work/devset_india_n20000_feats.parquet


In [ ]:
# --- Stage 3: fit the deploy LGB+XGB blend (no OOF file -> frozen t*=0.98) ----
FEATS = ROOT + "/dataset/work/devset_india_n20000_feats.parquet"
run("train.py", FEATS, "--neg", "3", "--prob", "blend")


In [ ]:
# --- Stage 4: score EVERY test S1 (all countries) -> the two TSVs ------------
import os
NCPU = os.cpu_count() or 4
run("infer.py", "--topk", "200", "--batch-s1", "5000", "--procs", str(NCPU))


In [ ]:
# --- Stage 5: validate the two deliverables (format + coverage, no GT needed) -
import os, pandas as pd
MRES = "/kaggle/working/matching_results.tsv"
CRES = "/kaggle/working/candidate_pairs.tsv"

s1 = set()
for ch in pd.read_csv(ROOT + "/dataset/clean/test/test_source1.tsv", sep="\t",
                      dtype=str, keep_default_na=False, na_values=[], quoting=3,
                      usecols=["entity_id"], chunksize=400000):
    s1.update(ch["entity_id"])

m = pd.read_csv(MRES, sep="\t", dtype=str, keep_default_na=False, na_values=[], quoting=3)
cd = pd.read_csv(CRES, sep="\t", dtype=str, keep_default_na=False, na_values=[], quoting=3)
assert list(m.columns) == ["source1_entity_id", "matched_entity_ids"], m.columns.tolist()
assert list(cd.columns) == ["source1_entity_id", "candidate_entity_ids"], cd.columns.tolist()

mset, cset = set(m["source1_entity_id"]), set(cd["source1_entity_id"])
cand_map = dict(zip(cd["source1_entity_id"], cd["candidate_entity_ids"]))
notsub = 0
for sid, mm in zip(m["source1_entity_id"], m["matched_entity_ids"]):
    pm = {x for x in mm.split(",") if x}
    pc = {x for x in cand_map.get(sid, "").split(",") if x}
    if not pm.issubset(pc):
        notsub += 1
nonempty = sum(1 for v in m["matched_entity_ids"] if v)

print("test S1 entities          :", len(s1))
print("matching rows / candidate rows:", len(m), "/", len(cd))
print("every test S1 in matching :", s1.issubset(mset), "(missing %d)" % len(s1 - mset))
print("every test S1 in candidates:", s1.issubset(cset), "(missing %d)" % len(s1 - cset))
print("rows w/ match NOT subset of candidates:", notsub)
print("non-empty matched rows    : %d (%.1f%%)" % (nonempty, 100 * nonempty / max(len(m), 1)))
ok = (s1.issubset(mset) and notsub == 0 and
      list(m.columns) == ["source1_entity_id", "matched_entity_ids"])
print("\nVALIDATION:", "PASS" if ok else "CHECK ABOVE")
print("OUTPUTS in /kaggle/working:", [f for f in os.listdir("/kaggle/working") if f.endswith(".tsv")])
